# P1-A0B: arithmetic competence revision

Research 002. Four-run, paired NumPy development screen. This is **not a history-sufficiency test**. Run cells in order: by default they prepare sources, test methods and show the recorded report. Full 24,000-update reproduction is explicitly opt-in. Code was executed locally; the notebook was not run in an interactive Colab session.

In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'
import ast, base64, hashlib, json, subprocess, sys, tempfile, time, zipfile
from pathlib import Path
from IPython.display import Markdown, display
ROOT=Path(tempfile.mkdtemp(prefix='p1_a0b_'))
SOURCES={'run_a0b.py': '"""P1-A0B fixed development panel; only NumPy plus the original project code."""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport platform\nimport sys\nimport time\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nsys.path.insert(0, str(ROOT / "prior"))\nimport experiment as base\nfrom src.tiny_autograd import Adam, cross_entropy\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef write_json(path, data):\n    Path(path).write_text(json.dumps(data, indent=2, allow_nan=False) + "\\n")\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef array_digest(value):\n    value = np.ascontiguousarray(value)\n    header = json.dumps([str(value.dtype), value.shape]).encode()\n    return hashlib.sha256(header + value.tobytes()).hexdigest()\n\n\ndef weight_digest(model):\n    h = hashlib.sha256()\n    for name, value in sorted(model.state_dict().items()):\n        h.update(name.encode())\n        h.update(value.tobytes())\n    return h.hexdigest()\n\n\ndef read_protocol():\n    return json.loads((ROOT / "P1_A0B_protocol.json").read_text())\n\n\nclass MatrixAdamW(Adam):\n    """Exact old Adam at lambda=0; otherwise decay matrices separately."""\n    def __init__(self, parameters, *, weight_decay, **kwargs):\n        super().__init__(parameters, **kwargs)\n        self.weight_decay = float(weight_decay)\n        self.decayed = [p for p in self.parameters if p.ndim >= 2]\n        if not 0 <= self.learning_rate * self.weight_decay < 1:\n            raise ValueError("Unsupported decay factor.")\n\n    def step(self):\n        if self.weight_decay:\n            factor = 1.0 - self.learning_rate * self.weight_decay\n            for p in self.decayed:\n                p.data *= factor\n        return super().step()\n\n\ndef validate_protocol(p):\n    assert p["study_id"] == "P1-A0B" and p["version"] == "1.0.0"\n    d = p["design"]\n    assert d["model_seeds"] == [2017, 2027]\n    assert d["arms"] == [{"name": "no_decay", "weight_decay": 0.0}, {"name": "matrix_decay", "weight_decay": 1.0}]\n    assert d["run_order"] == [[s, a["name"]] for s in d["model_seeds"] for a in d["arms"]]\n    assert d["updates_per_run"] == 6000 and d["maximum_total_updates"] == 24000\n    assert d["maximum_natural_training_runs"] == 4\n    assert d["checkpoints"] == [0, 1200, 3000, 6000]\n    assert not p["decisions"]["history_probes_allowed"] and not p["decisions"]["confirmation_allowed"]\n    old = base.read_protocol()\n    assert old["model"]["d_model"] == 24 and old["model"]["n_steps"] == 6\n    assert old["data"]["modulus"] == 17\n    for key in ("batch_size", "learning_rate", "beta1", "beta2", "epsilon"):\n        assert p["preserved"][key] == old["training"][key]\n    assert p["preserved"]["gradient_clip"] == old["training"]["max_grad_norm"]\n    assert p["measurements"]["competence_threshold"] == old["measurements"]["competence_gate"]\n    assert not set(d["model_seeds"]) & set(p["preserved"]["old_reserved_model_seeds"])\n    return old\n\n\ndef scientific_files():\n    required = ["P1_A0B_protocol.json", "P1_A0B_Protocol.md", "run_a0b.py", "checks_a0b.py",\n                "prior/experiment.py", "prior/test_methods.py", "prior/P1_A0_protocol.json",\n                "prior/P1_A0_Protocol.md", "prior/P1_A0_Report.md", "prior/vendor/T1_source_0.1.0.zip",\n                "prior/vendor/t1_original/src/__init__.py",\n                "prior/vendor/t1_original/src/tiny_autograd.py",\n                "prior/vendor/t1_original/src/tiny_transformer.py"]\n    return {name: digest(ROOT / name) for name in required}\n\n\ndef freeze(output, mode):\n    output = Path(output).resolve()\n    output.mkdir(parents=True, exist_ok=False)\n    frozen = {"frozen_at_utc": now(), "mode": mode, "files_sha256": scientific_files(),\n              "scope": "Local prospective audit, not independent registration."}\n    write_json(output / "FREEZE.json", frozen)\n    write_json(output / "FROZEN_PROTOCOL.json", read_protocol())\n    return output, frozen\n\n\ndef check_freeze(frozen):\n    assert scientific_files() == frozen["files_sha256"], "Frozen input changed."\n\n\ndef checkpoint(model, optimizer, update, datasets, out, old_protocol):\n    model.save(out / f"model_{update:04d}.npz")\n    if optimizer is not None:\n        moments = {f"m_{i}": v for i, v in enumerate(optimizer.m)}\n        moments.update({f"v_{i}": v for i, v in enumerate(optimizer.v)})\n        moments["step_count"] = np.array(optimizer.step_count)\n        np.savez_compressed(out / f"optimizer_{update:04d}.npz", **moments)\n    row = {"update": update, "recorded_at_utc": now()}\n    for name in ("train", "calibration"):\n        d = datasets[name]\n        tr = base.extract(model, d["tokens"], batch_size=64)\n        row[name] = base.measure(tr, d["labels"], d["family_ids"], old_protocol)\n        # Full states are exactly regenerable from the saved weights/input arrays.\n        np.savez_compressed(out / f"{name}_predictions_{update:04d}.npz", logits=tr["logits"], probabilities=tr["probabilities"])\n        if name == "calibration":\n            row["resume_errors"] = base.resume_audit(model, tr)\n    print(f"{out.name} checkpoint {update:4d}: train={row[\'train\'][\'accuracy_by_step\'][-1]:.4f}, "\n          f"calibration={row[\'calibration\'][\'accuracy_by_step\'][-1]:.4f}", flush=True)\n    return row\n\n\ndef run_one(seed, arm, batches, data, panel_out, protocol, old_protocol):\n    out = panel_out / f"seed_{seed}_{arm[\'name\']}"\n    out.mkdir(exist_ok=False)\n    model = base.ArithmeticTransformer(old_protocol, seed=seed)\n    initial = weight_digest(model)\n    optimizer = MatrixAdamW(model.parameters(), weight_decay=arm["weight_decay"],\n        learning_rate=protocol["preserved"]["learning_rate"], beta1=protocol["preserved"]["beta1"],\n        beta2=protocol["preserved"]["beta2"], epsilon=protocol["preserved"]["epsilon"],\n        max_grad_norm=protocol["preserved"]["gradient_clip"])\n    metadata = {"seed": seed, "arm": arm["name"], "weight_decay": arm["weight_decay"],\n        "initial_weight_sha256": initial, "batch_index_sha256": array_digest(batches),\n        "parameter_count": sum(p.data.size for p in model.parameters()),\n        "decayed_parameter_names": [p.name for p in optimizer.decayed],\n        "started_at_utc": now()}\n    write_json(out / "RUN_MANIFEST.json", metadata)\n    started = time.perf_counter()\n    checkpoints = [checkpoint(model, None, 0, data, out, old_protocol)]\n    write_json(out / "checkpoints.json", checkpoints)\n    metadata["training_started_at_utc"] = now()\n    write_json(out / "RUN_MANIFEST.json", metadata)\n    with (out / "training_log.jsonl").open("x") as log:\n        for update, batch in enumerate(batches, start=1):\n            _, logits, _ = model(data["train"]["tokens"][batch])\n            loss = cross_entropy(logits[-1], data["train"]["labels"][batch])\n            optimizer.zero_grad()\n            loss.backward()\n            if not np.isfinite(loss.data) or any(not np.all(np.isfinite(p.grad)) for p in optimizer.parameters):\n                raise FloatingPointError(f"Non-finite objective/gradient: {seed}/{arm[\'name\']}/{update}")\n            norm = optimizer.step()\n            if any(not np.all(np.isfinite(p.data)) for p in optimizer.parameters):\n                raise FloatingPointError("Non-finite weights.")\n            log.write(json.dumps({"update": update, "loss": float(loss.data), "grad_norm": norm,\n                                  "elapsed_s": time.perf_counter()-started}) + "\\n")\n            if update in protocol["design"]["checkpoints"]:\n                log.flush()\n                checkpoints.append(checkpoint(model, optimizer, update, data, out, old_protocol))\n                write_json(out / "checkpoints.json", checkpoints)\n            elif update % protocol["design"]["logging_interval"] == 0:\n                print(f"{out.name} update {update}/6000, batch NLL={float(loss.data):.4f}", flush=True)\n            if update % 25 == 0:\n                gc.collect()\n    final = checkpoints[-1]["calibration"]\n    eligible = [r["step"] for r in final["decision_change"] if r["eligible"]]\n    result = {**metadata, "completed_at_utc": now(), "updates_completed": len(batches),\n        "elapsed_seconds": time.perf_counter()-started, "checkpoints": checkpoints,\n        "final_train_accuracy": checkpoints[-1]["train"]["accuracy_by_step"][-1],\n        "final_calibration_accuracy": final["accuracy_by_step"][-1],\n        "competence_passed": final["accuracy_by_step"][-1] >= protocol["measurements"]["competence_threshold"],\n        "eligible_steps": eligible, "history_probes_fitted": 0}\n    write_json(out / "result.json", result)\n    return result\n\n\ndef decide(runs, protocol):\n    regimes = []\n    for arm in protocol["design"]["arms"]:\n        these = [r for r in runs if r["arm"] == arm["name"]]\n        complete = len(these) == 2 and all(r["updates_completed"] == 6000 for r in these)\n        common = sorted(set.intersection(*(set(r["eligible_steps"]) for r in these))) if these else []\n        competence = complete and all(r["competence_passed"] for r in these)\n        target = len(common) >= protocol["measurements"]["min_common_eligible_steps"]\n        regimes.append({"arm": arm["name"], "complete": complete, "competence_passed_all_seeds": competence,\n                        "common_eligible_steps": common, "target_passed": target,\n                        "ready_for_observer_design": bool(competence and target)})\n    selected = next((r["arm"] for r in regimes if r["ready_for_observer_design"]), None)\n    paired = []\n    for seed in protocol["design"]["model_seeds"]:\n        control = next(r for r in runs if r["seed"] == seed and r["arm"] == "no_decay")\n        decay = next(r for r in runs if r["seed"] == seed and r["arm"] == "matrix_decay")\n        assert control["initial_weight_sha256"] == decay["initial_weight_sha256"]\n        assert control["batch_index_sha256"] == decay["batch_index_sha256"]\n        paired.append({"seed": seed, "decay_minus_no_decay_accuracy":\n                       decay["final_calibration_accuracy"] - control["final_calibration_accuracy"]})\n    return {"regimens": regimes, "selected_development_candidate": selected, "paired_effects": paired,\n            "mean_paired_accuracy_difference": float(np.mean([r["decay_minus_no_decay_accuracy"] for r in paired])),\n            "status": "candidate_ready_for_observer_design" if selected else "no_regimen_ready",\n            "history_probes_fitted": 0, "confirmation_opened": False, "sufficiency_hypothesis_tested": False}\n\n\ndef run_panel(mode, destination):\n    protocol = read_protocol()\n    old = validate_protocol(protocol)\n    out, frozen = freeze(destination, mode)\n    from checks_a0b import run_checks\n    try:\n        checks = run_checks()\n        write_json(out / "method_checks.json", checks)\n        if mode == "methods":\n            check_freeze(frozen)\n            print(json.dumps(checks, indent=2))\n            return checks\n        started = time.perf_counter()\n        data, manifest = base.make_partitions(old)\n        write_json(out / "partitions.json", manifest)\n        assert manifest["example_counts"] == {"train": 2992, "calibration": 950}\n        np.savez_compressed(out / "development_data.npz", **{\n            f"{split}_{name}": values for split, d in data.items() for name, values in d.items()})\n        streams = {}\n        for seed in protocol["design"]["model_seeds"]:\n            streams[seed] = np.random.default_rng(500000 + seed).integers(\n                0, len(data["train"]["labels"]), size=(6000, 64), dtype=np.int64)\n            np.savez_compressed(out / f"batch_stream_{seed}.npz", indices=streams[seed])\n        runs = []\n        for seed, name in protocol["design"]["run_order"]:\n            check_freeze(frozen)\n            arm = next(a for a in protocol["design"]["arms"] if a["name"] == name)\n            runs.append(run_one(seed, arm, streams[seed], data, out, protocol, old))\n            write_json(out / "completed_runs.json", runs)\n        decision = decide(runs, protocol)\n        check_freeze(frozen)\n        result = {"study_id": protocol["study_id"], "version": protocol["version"],\n                  "completed_at_utc": now(), "elapsed_seconds": time.perf_counter()-started,\n                  "runs": runs, "decision": decision, "total_updates": sum(r["updates_completed"] for r in runs),\n                  "reserved_examples_evaluated": 0, "old_models_loaded": False,\n                  "frozen_sources_unchanged": True, "methods_passed": True,\n                  "environment": {"python": sys.version, "numpy": np.__version__, "platform": platform.platform()}}\n        write_json(out / "result.json", result)\n        print(json.dumps(decision, indent=2), flush=True)\n        return result\n    except Exception as exc:\n        write_json(out / "INCOMPLETE.json", {"time_utc": now(), "error": repr(exc),\n                                            "status": "incomplete_or_invalid_no_claim"})\n        raise\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("mode", choices=["methods", "panel"])\n    parser.add_argument("--output", required=True, type=Path)\n    args = parser.parse_args()\n    run_panel(args.mode, args.output)\n', 'checks_a0b.py': '"""Synthetic optimizer, pairing and inherited implementation checks; no natural fits."""\nimport numpy as np\n\nimport run_a0b as run\nfrom src.tiny_autograd import Adam, parameter\nfrom test_methods import run_checks as old_checks\n\n\ndef run_checks():\n    result = {"inherited_methods": old_checks()}\n    p = run.read_protocol()\n    old = run.validate_protocol(p)\n    rng = np.random.default_rng(9011)\n    # Zero-decay must be an EXACT extension of old Adam, including clipping/moments.\n    initial = [rng.normal(size=(2, 3)).astype(np.float32), rng.normal(size=3).astype(np.float32)]\n    a = [parameter(v.copy(), f"a{i}") for i, v in enumerate(initial)]\n    b = [parameter(v.copy(), f"b{i}") for i, v in enumerate(initial)]\n    adam = Adam(a, learning_rate=0.001)\n    adamw0 = run.MatrixAdamW(b, learning_rate=0.001, weight_decay=0)\n    for _ in range(10):\n        for aa, bb in zip(a, b):\n            gradient = rng.normal(size=aa.shape).astype(np.float32)\n            aa.grad[...] = gradient\n            bb.grad[...] = gradient\n        assert adam.step() == adamw0.step()\n        for aa, bb in zip(a, b):\n            np.testing.assert_array_equal(aa.data, bb.data)\n        for am, bm in zip(adam.m + adam.v, adamw0.m + adamw0.v):\n            np.testing.assert_array_equal(am, bm)\n    result["zero_decay_exact_original_adam_equivalence"] = True\n    # Pure shrink on matrices with zero gradients; vector must be unchanged.\n    matrix = parameter(np.array([[2.0, -3.0]], dtype=np.float32), "matrix")\n    vector = parameter(np.array([2.0, -3.0], dtype=np.float32), "vector")\n    opt = run.MatrixAdamW([matrix, vector], learning_rate=0.001, weight_decay=1)\n    opt.step()\n    np.testing.assert_allclose(matrix.data, np.array([[2.0, -3.0]]) * 0.999, atol=1e-7)\n    np.testing.assert_array_equal(vector.data, [2.0, -3.0])\n    result["zero_gradient_shrink_and_matrix_only_mask"] = True\n    # Multistep hand calculation with pre-decay gradients, global clipping and bias correction.\n    mats = [parameter(np.array([[1.5, -2.0]], dtype=np.float32), "w"),\n            parameter(np.array([0.25, -0.5], dtype=np.float32), "b")]\n    expected = [v.data.astype(np.float64).copy() for v in mats]\n    m, v = [np.zeros_like(x) for x in expected], [np.zeros_like(x) for x in expected]\n    opt = run.MatrixAdamW(mats, learning_rate=0.001, weight_decay=1)\n    for t in range(1, 6):\n        gradients = [rng.normal(size=x.shape) for x in expected]\n        for x, g in zip(mats, gradients):\n            x.grad[...] = g\n        norm = np.sqrt(sum(np.sum(g*g) for g in gradients))\n        for i, g in enumerate(gradients):\n            gg = g * min(1, 1/(norm+1e-12))\n            m[i] = 0.9*m[i] + 0.1*gg\n            v[i] = 0.999*v[i] + 0.001*gg*gg\n            if i == 0:\n                expected[i] *= 0.999\n            expected[i] -= 0.001*(m[i]/(1-0.9**t))/(np.sqrt(v[i]/(1-0.999**t))+1e-8)\n        opt.step()\n        for got, want in zip(mats, expected):\n            np.testing.assert_allclose(got.data, want, atol=5e-7, rtol=2e-6)\n    result["manual_multistep_decoupled_update"] = True\n    hashes = []\n    for seed in p["design"]["model_seeds"]:\n        a, b = [run.base.ArithmeticTransformer(old, seed=seed) for _ in range(2)]\n        assert run.weight_digest(a) == run.weight_digest(b)\n        hashes.append(run.weight_digest(a))\n        assert sum(x.data.size for x in a.parameters()) == 5873\n        batch_a = np.random.default_rng(500000+seed).integers(0,2992,size=(6000,64),dtype=np.int64)\n        batch_b = np.random.default_rng(500000+seed).integers(0,2992,size=(6000,64),dtype=np.int64)\n        np.testing.assert_array_equal(batch_a,batch_b)\n    assert len(set(hashes)) == 2\n    result["paired_initial_weights_and_full_batch_streams"] = True\n    data, partitions = run.base.make_partitions(old)\n    assert set(data) == {"train", "calibration"}\n    assert partitions["family_counts"] == {"train":581,"calibration":193,"reserve":195}\n    result["reserve_unexpanded_and_old_data_partition_preserved"] = True\n    # Decision fixture must reject one successful seed, and prefer simpler arm on ties.\n    fake = []\n    for seed in [2017,2027]:\n        for arm in ["no_decay","matrix_decay"]:\n            fake.append({"seed":seed,"arm":arm,"updates_completed":6000,\n                         "eligible_steps":[2,3],"competence_passed":True,\n                         "final_calibration_accuracy":0.95,\n                         "initial_weight_sha256":str(seed),"batch_index_sha256":str(seed)})\n    assert run.decide(fake,p)["selected_development_candidate"] == "no_decay"\n    fake[0]["competence_passed"] = False\n    assert run.decide(fake,p)["selected_development_candidate"] == "matrix_decay"\n    fake[1]["competence_passed"] = False\n    assert run.decide(fake,p)["selected_development_candidate"] is None\n    result["all_seed_gate_and_fixed_selection_policy"] = True\n    result["passed"] = True\n    return result\n\n\nif __name__ == "__main__":\n    import json\n    print(json.dumps(run_checks(),indent=2))\n', 'verify_a0b.py': '"""Recompute saved results and decisions without training or observer fitting."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nimport run_a0b as run\n\n\ndef verify(folder):\n    folder = Path(folder)\n    result = json.loads((folder / "result.json").read_text())\n    p = json.loads((folder / "FROZEN_PROTOCOL.json").read_text())\n    frozen = json.loads((folder / "FREEZE.json").read_text())\n    run.check_freeze(frozen)\n    assert p == run.read_protocol()\n    old = run.validate_protocol(p)\n    with np.load(folder / "development_data.npz", allow_pickle=False) as z:\n        data = {name: z[name] for name in z.files}\n    expected, manifest = run.base.make_partitions(old)\n    for split, values in expected.items():\n        for key, value in values.items():\n            np.testing.assert_array_equal(value, data[f"{split}_{key}"])\n    assert set(data) == {f"{s}_{k}" for s in expected for k in expected[s]}\n    assert manifest == json.loads((folder / "partitions.json").read_text())\n    assert not set(expected["train"]["family_ids"]) & set(expected["calibration"]["family_ids"])\n    assert [[r["seed"],r["arm"]] for r in result["runs"]] == p["design"]["run_order"]\n    verified_runs, max_error = [], 0.0\n    first_step_logs = {}\n    for recorded in result["runs"]:\n        seed, arm = recorded["seed"], recorded["arm"]\n        sub = folder / f"seed_{seed}_{arm}"\n        assert frozen["frozen_at_utc"] < recorded["started_at_utc"] <= recorded["training_started_at_utc"] < recorded["completed_at_utc"]\n        with np.load(folder / f"batch_stream_{seed}.npz", allow_pickle=False) as z:\n            batches = z["indices"]\n        assert batches.shape == (6000,64) and batches.dtype == np.int64\n        regenerated = np.random.default_rng(500000+seed).integers(0,2992,size=(6000,64),dtype=np.int64)\n        np.testing.assert_array_equal(regenerated,batches)\n        assert recorded["batch_index_sha256"] == run.array_digest(batches)\n        log = [json.loads(line) for line in (sub / "training_log.jsonl").read_text().splitlines()]\n        assert [r["update"] for r in log] == list(range(1,6001))\n        assert all(np.isfinite(r["loss"]) and np.isfinite(r["grad_norm"]) for r in log)\n        first_step_logs[(seed,arm)] = (log[0]["loss"], log[0]["grad_norm"])\n        assert [c["update"] for c in recorded["checkpoints"]] == [0,1200,3000,6000]\n        for checkpoint in recorded["checkpoints"]:\n            update = checkpoint["update"]\n            model = run.base.ArithmeticTransformer(old, seed=seed)\n            model.load(sub / f"model_{update:04d}.npz")\n            if update == 0:\n                assert run.weight_digest(model) == recorded["initial_weight_sha256"]\n            if update:\n                with np.load(sub / f"optimizer_{update:04d}.npz",allow_pickle=False) as z:\n                    assert int(z["step_count"]) == update\n                    assert len(z.files) == 2*len(model.parameters())+1\n                    for i,parameter in enumerate(model.parameters()):\n                        for prefix in ("m","v"):\n                            assert z[f"{prefix}_{i}"].shape == parameter.shape\n                            assert np.all(np.isfinite(z[f"{prefix}_{i}"]))\n                        assert np.all(z[f"v_{i}"] >= 0)\n            for split in ("train","calibration"):\n                labels, fids = data[f"{split}_labels"], data[f"{split}_family_ids"]\n                with np.load(sub / f"{split}_predictions_{update:04d}.npz",allow_pickle=False) as z:\n                    logits,probabilities = z["logits"],z["probabilities"]\n                # Full forward recomputation at every saved checkpoint, no fitting.\n                recreated = run.base.extract(model,data[f"{split}_tokens"])\n                error = float(np.max(np.abs(recreated["logits"]-logits)))\n                max_error = max(max_error,error)\n                np.testing.assert_allclose(recreated["logits"],logits,atol=2e-5,rtol=2e-5)\n                np.testing.assert_allclose(probabilities,run.base.numpy_softmax(logits),atol=2e-6,rtol=2e-6)\n                predictions = logits.argmax(-1)\n                accuracies = (predictions == labels[:,None]).mean(0)\n                np.testing.assert_array_equal(accuracies,checkpoint[split]["accuracy_by_step"])\n                for row in checkpoint[split]["decision_change"]:\n                    changed = predictions[:,row["step"]] != predictions[:,-1]\n                    counts = [int((~changed).sum()),int(changed.sum())]\n                    families = [len(np.unique(fids[~changed])),len(np.unique(fids[changed]))]\n                    assert counts == [row["unchanged_examples"],row["changed_examples"]]\n                    assert families == [row["unchanged_families"],row["changed_families"]]\n                    gate = min(counts)>=50 and min(families)>=25 and min(counts)/sum(counts)>=0.1\n                    assert gate == row["eligible"]\n                if split == "calibration":\n                    errors = run.base.resume_audit(model,recreated)\n                    for e in errors.values():\n                        assert e <= 2e-5 + 2e-5*float(np.max(np.abs(logits)))\n            assert max(checkpoint["resume_errors"].values()) <= 2e-5 + 2e-5*float(np.max(np.abs(logits)))\n        verified_runs.append([seed,arm])\n    for seed in (2017,2027):\n        assert first_step_logs[(seed,"no_decay")] == first_step_logs[(seed,"matrix_decay")]\n    # Independent decision reconstruction; do not call the production decide().\n    ready, recomputed_regimes = [], []\n    for name in ("no_decay","matrix_decay"):\n        group = [r for r in result["runs"] if r["arm"]==name]\n        competent = all(r["checkpoints"][-1]["calibration"]["accuracy_by_step"][-1]>=0.9 for r in group)\n        common = set([2,3,4,5])\n        for r in group:\n            rows = r["checkpoints"][-1]["calibration"]["decision_change"]\n            common &= {row["step"] for row in rows if row["eligible"]}\n        passed = competent and len(common)>=2\n        recorded = next(x for x in result["decision"]["regimens"] if x["arm"]==name)\n        assert recorded["competence_passed_all_seeds"] == competent\n        assert recorded["common_eligible_steps"] == sorted(common)\n        assert recorded["ready_for_observer_design"] == passed\n        if passed:\n            ready.append(name)\n        recomputed_regimes.append({"arm":name,"ready":passed})\n    assert result["decision"]["selected_development_candidate"] == (ready[0] if ready else None)\n    differences = []\n    for row in result["decision"]["paired_effects"]:\n        subset = {r["arm"]:r for r in result["runs"] if r["seed"]==row["seed"]}\n        expected_diff = subset["matrix_decay"]["final_calibration_accuracy"] - subset["no_decay"]["final_calibration_accuracy"]\n        assert expected_diff == row["decay_minus_no_decay_accuracy"]\n        differences.append(expected_diff)\n    assert float(np.mean(differences)) == result["decision"]["mean_paired_accuracy_difference"]\n    assert result["total_updates"] == 24000\n    assert result["reserved_examples_evaluated"] == result["decision"]["history_probes_fitted"] == 0\n    run.check_freeze(frozen)\n    return {"passed":True,"verified_runs":verified_runs,"saved_checkpoints_verified":16,\n            "all_train_and_calibration_logits_recomputed":True,"max_logit_error":max_error,\n            "all_accuracy_flip_counts_and_gates_recomputed":True,"paired_initializations_streams_and_first_gradient_verified":True,\n            "all_24000_updates_and_optimizer_moments_verified":True,"reserved_examples_absent":True,\n            "scientific_freeze_preceded_all_training_and_is_unchanged":True,\n            "regimen_decisions_recomputed":recomputed_regimes,\n            "scope":"Separate numerical audit; not independent model replication."}\n\n\nif __name__ == "__main__":\n    import sys\n    folder = Path(sys.argv[1])\n    result = verify(folder)\n    run.write_json(folder / "result_verification.json",result)\n    print(json.dumps(result,indent=2))\n', 'P1_A0B_protocol.json': '{\n  "study_id": "P1-A0B",\n  "version": "1.0.0",\n  "date_utc": "2026-09-15",\n  "title": "Arithmetic competence: paired training-budget and weight-decay development screen",\n  "scope": "A new bounded development stage informed by P1-A0; not confirmation or a history-observer study.",\n  "rationale": "P1-A0 had a large train/calibration gap after 1200 updates. Test longer training with a paired decoupled-weight-decay contrast, retaining task, architecture and exposed development split.",\n  "preserved": {\n    "task": "(a+b+c) mod 17",\n    "architecture": "P1-A0 six-step weight-tied encoder; width24 heads2 feedforward48; 5873 parameters",\n    "data_source": "Exact P1-A0 canonical-family split and train/calibration examples; these calibration outcomes are exposed development data, not untouched confirmation.",\n    "train_families": 581,\n    "train_examples": 2992,\n    "calibration_families": 193,\n    "calibration_examples": 950,\n    "reserved_families": 195,\n    "reserved_example_policy": "Do not expand, train on, or evaluate reserved families.",\n    "old_reserved_model_seeds": [1103, 1201, 1301],\n    "loss": "Final-step cross entropy only; no intermediate supervision",\n    "batch_size": 64,\n    "learning_rate": 0.001,\n    "beta1": 0.9,\n    "beta2": 0.999,\n    "epsilon": 0.00000001,\n    "gradient_clip": 1.0\n  },\n  "design": {\n    "model_seeds": [2017, 2027],\n    "arms": [{"name": "no_decay", "weight_decay": 0.0}, {"name": "matrix_decay", "weight_decay": 1.0}],\n    "run_order": [[2017, "no_decay"], [2017, "matrix_decay"], [2027, "no_decay"], [2027, "matrix_decay"]],\n    "updates_per_run": 6000,\n    "maximum_natural_training_runs": 4,\n    "maximum_total_updates": 24000,\n    "sampling_seed_rule": "500000 + model_seed; NumPy PCG64 with-replacement index matrix [6000,64], int64",\n    "pairing": "Identical initial weights and complete batch-index matrix within each model-seed pair; train both arms from initialization, not the old checkpoint.",\n    "checkpoints": [0, 1200, 3000, 6000],\n    "selection_checkpoint": 6000,\n    "logging_interval": 1000,\n    "evaluation_batch_size": 64,\n    "run_all_planned_arms": true,\n    "best_checkpoint_selection": false,\n    "hyperparameter_search": false,\n    "outcome_driven_extension": false\n  },\n  "optimizer": {\n    "implementation": "Original P1-A0 Adam plus decoupled multiplicative decay of parameter matrices BEFORE the Adam update, using gradients computed at pre-decay parameters.",\n    "decay_parameters": "Every trainable tensor with ndim >= 2: token and position embeddings and linear weights. Exclude all biases and LayerNorm scale/bias vectors.",\n    "matrix_update": "theta_new=(1-lr*lambda)*theta_old - lr*m_hat/(sqrt(v_hat)+epsilon)",\n    "bias_and_norm_update": "Original Adam only",\n    "decay_step_fraction": 0.001,\n    "justification": "Weight decay and longer training have precedent in algorithmic generalization research. Lambda=1 is a fixed literature-motivated development candidate, not an optimum; matrix-only decay and the smaller tied architecture differ from that literature."\n  },\n  "measurements": {\n    "primary_contrast": "Final calibration exact-answer accuracy: matrix_decay minus no_decay, paired by model seed; report both differences and their arithmetic mean descriptively.",\n    "budget_diagnostic": "Within each arm/seed, report accuracy at 1200,3000,6000 without selecting a best checkpoint.",\n    "competence_threshold": 0.9,\n    "competence_gate_rationale": "Same declared development readiness threshold as P1-A0, not a validated deployment criterion.",\n    "decision_target": "Z_l=1[argmax(logits_l) != argmax(logits_6)]",\n    "candidate_steps": [2, 3, 4, 5],\n    "min_examples_per_class": 50,\n    "min_families_per_class": 25,\n    "min_class_fraction": 0.1,\n    "min_common_eligible_steps": 2,\n    "no_inferential_p_values": true,\n    "no_model_population_confidence_interval": true,\n    "descriptive_limit": "Two new initialization seeds on the same exposed development families do not establish out-of-sample model/population generalization or an untouched replication."\n  },\n  "decisions": {\n    "arm_ready": "Both seeds complete all updates, final accuracy >=90% for both, and at least two of steps2..5 satisfy all target-count/family/prevalence gates for BOTH seeds.",\n    "arm_preference_if_both_ready": "no_decay (simpler unchanged optimizer)",\n    "one_ready": "The passing arm is a development candidate for later observer-protocol design, not automatic probe fitting or confirmation.",\n    "none_ready": "Close this bounded competence screen with no ready regimen; do not add updates, select a best seed, or fit history predictors.",\n    "numeric_or_method_failure": "Mark study incomplete/invalid, preserve diagnostics, stop; no replacement run within this protocol.",\n    "history_probes_allowed": false,\n    "confirmation_allowed": false,\n    "sufficiency_hypothesis_tested": false,\n    "next_boundary": "Any further competence recipe or observer study requires its own justification and frozen protocol; retain every current failure and the sealed old/reserved data."\n  },\n  "artifacts": {\n    "save": ["local_protocol_source_freeze", "exact_development_arrays", "paired_batch_index_matrices", "all_scheduled_model_checkpoints", "all_scheduled_train_and_calibration_logits", "optimizer_moments_and_step_counts", "all_update_logs", "resume_checks", "regimen_decisions", "verification_report"],\n    "full_states": "Regenerate from frozen weights and exact inputs to keep the archive compact. No observer fitting is performed.",\n    "notebook": "Standalone NumPy Colab; method checks and recorded report by default; full development reproduction is explicitly opt-in."\n  },\n  "sources": ["P1_A0_Report.md", "P1_A0_protocol.json", "https://arxiv.org/abs/2201.02177", "https://arxiv.org/abs/1711.05101"]\n}\n', 'P1_A0B_Protocol.md': "# P1-A0B — Arithmetic competence revision\n\nResearch 002 · 15 September 2026 · v1.0.0  \nProspective, bounded development screen; no history probes or confirmation.\n\n## Why this stage\n\nP1-A0 finished 1,200 updates with 68.11% training accuracy and 3.16% accuracy on\nunseen canonical operand combinations. That gap motivates a competence revision,\nnot a present-state-sufficiency conclusion. We now test a longer training budget\nand one fixed regularization contrast. Earlier T1/T2 and P1-A0 findings remain\nunchanged. P1-A1 remains the future observer stage; this is still readiness work.\n\nSmall algorithmic tasks can show late generalization, and weight decay can help\nin some settings. The original Grokking paper used much larger models and budgets\nthan ours: its appendix describes AdamW with lr=0.001, decay=1 and up to 100,000\nupdates for most experiments. This is motivation, not a claim that its results\ntransfer here or that 6,000 updates are sufficient. We preserve our beta2=0.999\nand use matrix-only decay, so this is not a reproduction of that recipe.\n\n## Frozen contrast\n\nKeep `(a+b+c) mod 17`, three operand tokens plus CLS, width24, two attention heads,\nfeed-forward width48, and one pre-norm Transformer block reused for six steps.\nThere are 5,873 trainable parameters. Keep final-only cross entropy, batch64,\nlr0.001, Adam beta1=0.9/beta2=0.999, epsilon1e-8 and global gradient clip1.0.\n\n| Setting | No-decay arm | Matrix-decay arm |\n|---|---|---|\n| Training updates | 6,000 | 6,000 |\n| Model seeds | 2017 and 2027 | Same paired seeds |\n| Batch stream | Fixed per seed | Identical paired stream |\n| Decoupled weight decay | 0 | 1, on matrices only |\n| Checkpoints | 0, 1,200, 3,000, 6,000 | Same |\n\nAll four runs start from initialization. No prior trained weights are loaded.\nWithin each seed, initial weights and all 6,000 batches must match exactly across\narms. This isolates the specified weight-decay change at the matched budget;\nscheduled checkpoints describe learning with additional updates. Do not interpret\ncomparisons with P1-A0's different seed as a controlled optimizer contrast.\n\nFor decayed matrices, the optimizer uses\n`theta_next = (1 - lr*decay)*theta - lr*m_hat/(sqrt(v_hat)+epsilon)`,\nwith moments from the globally clipped gradients at the pre-decay weights.\nThe multiplicative shrink is 0.1% per update, not 100%. Token/position embeddings\nand linear weight matrices are decayed; biases and LayerNorm vectors are not.\nBoth arms include the same loop; setting decay to zero must be numerically\nidentical to the original Adam implementation. Unit fixtures check this, the\nclosed-form first and multiple updates, and the matrix/vector mask.\n\n## Development data, not a new test set\n\nReuse exactly P1-A0's 2,992 training examples/581 families and 950 calibration\nexamples/193 families. All permutations of a sorted triple remain in one split.\nThese calibration outcomes already influenced the research direction; they are\n**exposed development data**, although the models never train on their labels.\nTwo new model seeds are not untouched study confirmation on fresh data.\n\nThe 195 reserved families remain unexpanded and unevaluated. Do not reshuffle\nthem into training. Old reserved model seeds 1103/1201/1301 remain unused.\nThe model-seed-specific stream seed is `500000 + seed`; save the entire int64\nbatch-index matrix and its hash. Input labels are exact arithmetic targets.\n\n## Outcomes fixed before training\n\nThe primary descriptive comparison is the 6,000-update calibration accuracy\ndifference, decay minus no-decay, for each seed and their mean. Report all four\nruns, all scheduled checkpoints, and training/calibration accuracy. No best\ncheckpoint, best seed, p-value, or two-seed model-population interval is used.\n\nA regimen is ready for future observer design only if BOTH model seeds reach\n90% final calibration accuracy and at least two **common** steps among 2..5 have:\n\n- At least 50 examples in each decision-change class.\n- At least 25 distinct families contributing to each class.\n- At least 10% of examples in each class.\n\nThe target is disagreement of the current shared-head argmax with the final\nargmax, not mathematical correctness. A family can contribute to both classes\nthrough different permutations. These gates avoid a near-constant target and\nspurious support from permutations; they are not a power analysis.\n\nUse only the checkpoint at update6,000 for readiness. If both regimens pass,\nprefer the no-decay regimen as the simpler unchanged optimizer. If neither\npasses, close this bounded screen without adding runs or updates. A single\nsuccessful seed cannot compensate for a failed seed. A numeric/method failure\ninvalidates completion and stops the panel; do not replace it with a new seed.\n\n## Scope and resources\n\nThe development budget is exactly four natural training runs, 24,000 updates,\nand no observer fits. Run order: seed2017 no-decay, seed2017 decay, seed2027\nno-decay, seed2027 decay. Log progress at 1,000-update intervals. No schedule or\nhyperparameter change is allowed based on emerging outcomes. New paths prevent\noverwriting prior outputs; a local freeze binds scientific inputs before fitting.\nThis local audit trail is not an independently timestamped registration service.\n\nRecheck original method fixtures, verify the new optimizer with synthetic tests,\nand audit continuation from all-token states at every checkpoint. Resuming the\nactual network from its current state remains an architectural property, not an\nawareness finding or evidence for cheap prediction. Save logits/checkpoints;\nregenerate full states when needed to keep the archive small.\n\nCompletion permits a decision about the **next design**, never automatic history\ntests. Future observer fitting still needs explicit predictor budgets, leakage\ncontrols, strong current-only references, a useful-prediction criterion and\nmargin, sensitivity tests through the actual fitted pipeline, family-aware\nuncertainty and a genuinely untouched confirmation plan. No claim about awareness,\nmetacognition, present-state sufficiency or the benefit of history follows here.\n\n## References\n\n- Prior sources: bundled unchanged `prior/P1_A0_Report.md` and\n  `prior/P1_A0_protocol.json`; original NumPy Transformer/autodiff is unchanged.\n- [Power et al., Grokking](https://arxiv.org/abs/2201.02177), especially sections3.1,\n  3.3 and appendixA.1.2: motivates longer optimization/regularization but uses a\n  different regime and does not establish our hypothesis.\n- [Loshchilov and Hutter, Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101):\n  supports separating weight decay from the adaptive gradient update.\n", 'prior/experiment.py': '"""P1-A0: bounded arithmetic readiness pilot; no history-probe fitting.\n\nRun: python experiment.py methods --output /new/output/path\n     python experiment.py pilot --output /different/new/output/path\nOnly NumPy and the Python standard library are required.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom contextlib import contextmanager\nfrom dataclasses import asdict\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport itertools\nimport json\nimport platform\nfrom pathlib import Path\nimport shutil\nimport sys\nimport time\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nsys.path.insert(0, str(ROOT / "vendor" / "t1_original"))\nfrom src.tiny_autograd import Adam, Tensor, cross_entropy\nfrom src.tiny_transformer import ModelConfig, TinyTransformer, numpy_softmax\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef read_protocol():\n    return json.loads((ROOT / "P1_A0_protocol.json").read_text())\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\nclass ArithmeticTransformer(TinyTransformer):\n    """Original Transformer block, reused at every computation step."""\n    def __init__(self, protocol=None, *, seed=None):\n        protocol = read_protocol() if protocol is None else protocol\n        spec, data = protocol["model"], protocol["data"]\n        config = ModelConfig(\n            vocab_size=data["modulus"] + 1,\n            sequence_length=data["operands"] + 1,\n            d_model=spec["d_model"], n_heads=spec["n_heads"],\n            d_ff=spec["d_ff"], n_layers=1, n_classes=data["modulus"],\n            layer_norm_epsilon=spec["layer_norm_epsilon"],\n        )\n        model_seed = spec["seed"] if seed is None else seed\n        if model_seed in protocol["scope"]["old_reserved_seeds_forbidden"]:\n            raise ValueError("The old reserved seeds remain sealed.")\n        super().__init__(config, model_seed)\n        self.refinements = spec["n_steps"]\n\n    def __call__(self, tokens):\n        if tokens.ndim != 2 or tokens.shape[1] != self.config.sequence_length:\n            raise ValueError("Expected [batch, CLS + operands].")\n        if not np.issubdtype(tokens.dtype, np.integer):\n            raise ValueError("Token IDs must be integers.")\n        if np.any(tokens < 0) or np.any(tokens >= self.config.vocab_size):\n            raise ValueError("Token ID outside vocabulary.")\n        one_hot = np.eye(self.config.vocab_size, dtype=np.float32)[tokens]\n        state = Tensor(one_hot) @ self.token_embedding + self.position_embedding\n        states, logits, attentions = [state], [self._read(state)], []\n        for _ in range(self.refinements):\n            state, attention = self.layers[0](state)\n            states.append(state)\n            logits.append(self._read(state))\n            attentions.append(attention)\n        return states, logits, attentions\n\n    def resume(self, state, step):\n        """Continue from all tokens; weights and the step index stay fixed."""\n        if not 0 <= step <= self.refinements:\n            raise ValueError("Invalid computation step.")\n        expected = (self.config.sequence_length, self.config.d_model)\n        if np.asarray(state).ndim != 3 or tuple(state.shape[1:]) != expected:\n            raise ValueError("Resume requires complete all-token state.")\n        current = Tensor(np.asarray(state, dtype=np.float32).copy())\n        for _ in range(step, self.refinements):\n            current, _ = self.layers[0](current)\n        return self._read(current).data.copy()\n\n    def load(self, path):\n        with np.load(path, allow_pickle=False) as saved:\n            params = {p.name: p for p in self.parameters()}\n            if set(saved.files) != set(params):\n                raise ValueError("Checkpoint key mismatch.")\n            for name, parameter in params.items():\n                if parameter.shape != saved[name].shape:\n                    raise ValueError("Checkpoint shape mismatch.")\n                parameter.data[...] = saved[name]\n\n\n@contextmanager\ndef inference(model):\n    """Avoid autograd graphs without modifying source or parameter values."""\n    parameters = model.parameters()\n    flags = [p.requires_grad for p in parameters]\n    try:\n        for p in parameters:\n            p.requires_grad = False\n        yield\n    finally:\n        for p, flag in zip(parameters, flags):\n            p.requires_grad = flag\n\n\ndef make_partitions(protocol=None):\n    """Expand train/calibration only; reserve rows are never instantiated."""\n    protocol = read_protocol() if protocol is None else protocol\n    spec = protocol["data"]\n    families = list(itertools.combinations_with_replacement(\n        range(spec["modulus"]), spec["operands"]))\n    order = np.random.default_rng(spec["partition_seed"]).permutation(len(families))\n    n_train = int(spec["train_fraction"] * len(families))\n    n_cal = int(spec["calibration_fraction"] * len(families))\n    ids = {"train": order[:n_train], "calibration": order[n_train:n_train+n_cal],\n           "reserve": order[n_train+n_cal:]}\n    datasets = {}\n    for name in ("train", "calibration"):\n        tokens, labels, family_ids = [], [], []\n        for family_id in sorted(ids[name].tolist()):\n            for operands in sorted(set(itertools.permutations(families[family_id]))):\n                tokens.append([spec["modulus"], *operands])\n                labels.append(sum(operands) % spec["modulus"])\n                family_ids.append(family_id)\n        datasets[name] = {\n            "tokens": np.asarray(tokens, dtype=np.int64),\n            "labels": np.asarray(labels, dtype=np.int64),\n            "family_ids": np.asarray(family_ids, dtype=np.int64),\n        }\n    manifest = {"family_ids": {k: v.tolist() for k, v in ids.items()},\n                "family_counts": {k: len(v) for k, v in ids.items()},\n                "example_counts": {k: len(v["labels"]) for k, v in datasets.items()},\n                "reserve_examples_constructed": False,\n                "total_canonical_families": len(families)}\n    return datasets, manifest\n\n\ndef extract(model, tokens, batch_size=64):\n    states, logits = [], []\n    with inference(model):\n        for start in range(0, len(tokens), batch_size):\n            ss, ll, _ = model(tokens[start:start+batch_size])\n            states.append(np.stack([s.data for s in ss], axis=1))\n            logits.append(np.stack([v.data for v in ll], axis=1))\n    state_array = np.concatenate(states)\n    logit_array = np.concatenate(logits)\n    return {"states": state_array, "logits": logit_array,\n            "probabilities": numpy_softmax(logit_array)}\n\n\ndef observable_features(trajectory, step, view="all_token", history=False):\n    """Extraction interface only: never accesses labels or future steps."""\n    if view not in {"cls", "all_token"}:\n        raise ValueError("Unknown observation view.")\n    if not 1 <= step < trajectory["states"].shape[1] - 1:\n        raise ValueError("Use an intermediate step.")\n\n    def at(index, chosen_view):\n        # Slice BEFORE calculations: changing future arrays cannot change features.\n        state = trajectory["states"][:, index]\n        probability = trajectory["probabilities"][:, index]\n        entropy = -(probability * np.log(np.clip(probability, 1e-8, 1))).sum(1, keepdims=True)\n        ordered = np.sort(probability, axis=1)\n        margin = (ordered[:, -1] - ordered[:, -2])[:, None]\n        values = state[:, 0] if chosen_view == "cls" else state.reshape(len(state), -1)\n        return np.concatenate([values, probability, entropy, margin], axis=1)\n\n    current = at(step, view)\n    if not history or step == 1:\n        return current\n    past = [at(index, "cls") for index in range(1, step)]\n    return np.concatenate([current, *past], axis=1)\n\n\ndef resume_audit(model, trajectory, count=16):\n    result = {}\n    expected = trajectory["logits"][:count, -1]\n    with inference(model):\n        for step in range(model.refinements + 1):\n            resumed = model.resume(trajectory["states"][:count, step], step)\n            error = float(np.max(np.abs(resumed - expected)))\n            if not np.allclose(resumed, expected, atol=2e-5, rtol=2e-5):\n                raise AssertionError(f"State resumption failed at step {step}: {error}")\n            result[str(step)] = error\n    return result\n\n\ndef measure(trajectory, labels, family_ids, protocol):\n    predictions = trajectory["logits"].argmax(axis=-1)\n    accuracy = (predictions == labels[:, None]).mean(axis=0)\n    rows = []\n    spec = protocol["measurements"]\n    for step in spec["eligible_candidate_steps"]:\n        flips = predictions[:, step] != predictions[:, -1]\n        counts = [int(np.sum(flips == value)) for value in (False, True)]\n        families = [len(np.unique(family_ids[flips == value])) for value in (False, True)]\n        eligible = (min(counts) >= spec["target_gate_min_events_per_class"] and\n                    min(families) >= spec["target_gate_min_families_per_class"] and\n                    min(counts) / len(flips) >= spec["target_gate_min_class_fraction"])\n        rows.append({"step": step, "unchanged_examples": counts[0], "changed_examples": counts[1],\n                     "unchanged_families": families[0], "changed_families": families[1],\n                     "flip_prevalence": float(flips.mean()), "eligible": bool(eligible)})\n    return {"accuracy_by_step": accuracy.tolist(), "decision_change": rows}\n\n\ndef freeze(output, mode):\n    output = Path(output).resolve()\n    output.mkdir(parents=True, exist_ok=False)  # refuse to overwrite a prior run\n    files = ["P1_A0_protocol.json", "P1_A0_Protocol.md", "experiment.py", "test_methods.py",\n             "vendor/T1_source_0.1.0.zip", "vendor/t1_original/src/tiny_autograd.py",\n             "vendor/t1_original/src/tiny_transformer.py"]\n    record = {"frozen_at_utc": utc_now(), "mode": mode,\n              "files_sha256": {name: sha256(ROOT / name) for name in files},\n              "scope": "Local prospective freeze, not independent registration."}\n    write_json(output / "FREEZE.json", record)\n    shutil.copyfile(ROOT / "P1_A0_protocol.json", output / "FROZEN_PROTOCOL.json")\n    return output, record\n\n\ndef verify_freeze(record):\n    for relative, expected in record["files_sha256"].items():\n        if sha256(ROOT / relative) != expected:\n            raise AssertionError(f"Frozen scientific input changed: {relative}")\n\n\ndef checkpoint(model, optimizer, update, datasets, output, protocol):\n    model.save(output / f"model_update_{update:04d}.npz")\n    if optimizer is not None:\n        moments = {f"m_{i}": m for i, m in enumerate(optimizer.m)}\n        moments.update({f"v_{i}": v for i, v in enumerate(optimizer.v)})\n        moments["step_count"] = np.array(optimizer.step_count)\n        np.savez_compressed(output / f"optimizer_update_{update:04d}.npz", **moments)\n    row = {"update": update, "recorded_at_utc": utc_now()}\n    for name in ("train", "calibration"):\n        data = datasets[name]\n        trajectory = extract(model, data["tokens"], protocol["training"]["evaluation_batch_size"])\n        row[name] = measure(trajectory, data["labels"], data["family_ids"], protocol)\n        if name == "calibration":\n            row["resume_errors"] = resume_audit(model, trajectory)\n            np.savez_compressed(output / f"calibration_trajectory_{update:04d}.npz", **trajectory)\n    print(f"checkpoint {update:4d}: train={row[\'train\'][\'accuracy_by_step\'][-1]:.4f} "\n          f"calibration={row[\'calibration\'][\'accuracy_by_step\'][-1]:.4f}", flush=True)\n    return row\n\n\ndef run(mode, output):\n    protocol = read_protocol()\n    output, frozen = freeze(output, mode)\n    from test_methods import run_checks\n    try:\n        checks = run_checks()\n        write_json(output / "method_checks.json", checks)\n        if mode == "methods":\n            verify_freeze(frozen)\n            print(json.dumps(checks, indent=2), flush=True)\n            return checks\n        if mode != "pilot":\n            raise ValueError("Unknown mode.")\n        started = time.perf_counter()\n        datasets, partitions = make_partitions(protocol)\n        write_json(output / "partitions.json", partitions)\n        np.savez_compressed(output / "development_data.npz", **{\n            f"{name}_{key}": value for name, data in datasets.items() for key, value in data.items()\n        })\n        model = ArithmeticTransformer(protocol)\n        spec = protocol["training"]\n        optimizer = Adam(model.parameters(), learning_rate=spec["learning_rate"],\n                         beta1=spec["beta1"], beta2=spec["beta2"], epsilon=spec["epsilon"],\n                         max_grad_norm=spec["max_grad_norm"])\n        rng = np.random.default_rng(spec["sampling_seed"])\n        checkpoints = [checkpoint(model, None, 0, datasets, output, protocol)]\n        write_json(output / "checkpoints.json", checkpoints)\n        train = datasets["train"]\n        training_started = utc_now()\n        with (output / "training_log.jsonl").open("x") as log:\n            for update in range(1, spec["updates"] + 1):\n                batch = rng.integers(0, len(train["labels"]), size=spec["batch_size"])\n                _, logits, _ = model(train["tokens"][batch])\n                loss = cross_entropy(logits[-1], train["labels"][batch])\n                optimizer.zero_grad()\n                loss.backward()\n                if not np.isfinite(loss.data) or any(not np.all(np.isfinite(p.grad)) for p in optimizer.parameters):\n                    raise FloatingPointError(f"Non-finite objective/gradient at update {update}")\n                norm = optimizer.step()\n                if any(not np.all(np.isfinite(p.data)) for p in optimizer.parameters):\n                    raise FloatingPointError(f"Non-finite parameters at update {update}")\n                log.write(json.dumps({"update": update, "loss": float(loss.data),\n                                      "grad_norm": norm, "elapsed_s": time.perf_counter()-started}) + "\\n")\n                if update in spec["checkpoints"]:\n                    log.flush()\n                    checkpoints.append(checkpoint(model, optimizer, update, datasets, output, protocol))\n                    write_json(output / "checkpoints.json", checkpoints)\n                elif update % spec["progress_every"] == 0:\n                    print(f"update {update}/{spec[\'updates\']}, batch NLL={float(loss.data):.4f}", flush=True)\n                if update % 25 == 0:\n                    gc.collect()\n        final = checkpoints[-1]["calibration"]\n        competence = final["accuracy_by_step"][-1] >= protocol["measurements"]["competence_gate"]\n        eligible = [r["step"] for r in final["decision_change"] if r["eligible"]]\n        target = len(eligible) >= protocol["measurements"]["target_gate_min_eligible_steps"]\n        status = ("competence_failed" if not competence else\n                  "target_failed" if not target else "ready_for_observer_design")\n        verify_freeze(frozen)\n        result = {\n            "study_id": protocol["study_id"], "version": protocol["version"],\n            "status": status, "methods_passed": True,\n            "seed": protocol["model"]["seed"], "updates_completed": spec["updates"],\n            "training_started_at_utc": training_started, "completed_at_utc": utc_now(),\n            "elapsed_seconds": time.perf_counter()-started,\n            "parameter_count": sum(p.data.size for p in model.parameters()),\n            "competence_passed": bool(competence), "target_passed": bool(target),\n            "eligible_steps": eligible, "final_calibration_accuracy": final["accuracy_by_step"][-1],\n            "checkpoints": checkpoints, "partitions": partitions,\n            "history_probes_fitted": 0, "reserved_examples_evaluated": 0,\n            "old_models_loaded": False, "confirmation_opened": False,\n            "sufficiency_hypothesis_tested": False,\n            "frozen_inputs_unchanged": True,\n            "environment": {"python": sys.version, "numpy": np.__version__, "platform": platform.platform()},\n            "model_config": asdict(model.config), "refinement_steps": model.refinements,\n        }\n        write_json(output / "result.json", result)\n        print(f"P1-A0 complete: {status}; no history observers were fitted.", flush=True)\n        return result\n    except Exception as exc:\n        write_json(output / "INCOMPLETE.json", {"time_utc": utc_now(), "error": repr(exc),\n                   "status": "invalid_or_incomplete_no_scientific_interpretation"})\n        raise\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("mode", choices=["methods", "pilot"])\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    run(args.mode, args.output)\n', 'prior/P1_A0_protocol.json': '{\n  "study_id": "P1-A0",\n  "version": "1.0.0",\n  "title": "Present-state sufficiency after arithmetic practice: readiness pilot",\n  "stage": "development_only_no_observer_fits",\n  "created_date_utc": "2026-09-15",\n  "question": "Can an affordable observer use a practiced arithmetic Transformer\'s current representation to predict future decision change without a useful increment from its computational history?",\n  "scope": {\n    "this_stage": "Validate code, exact continuation, held-out arithmetic competence and non-degenerate decision-change targets in one fixed pilot.",\n    "history_hypothesis_tested": false,\n    "observer_fitting_allowed": false,\n    "old_models_loaded": false,\n    "old_reserved_seeds_forbidden": [1103, 1201, 1301],\n    "maximum_natural_training_runs": 1,\n    "outcome_driven_reruns_or_extensions": false,\n    "confirmation": "Not opened. A subsequent observer protocol and new-model replication plan must be frozen separately."\n  },\n  "data": {\n    "task": "(a+b+c) mod 17",\n    "modulus": 17,\n    "operands": 3,\n    "input_tokens": "[CLS, a, b, c], where CLS has token id 17",\n    "family_key": "sorted operand triple: all permutations share a partition",\n    "partition_seed": 271828,\n    "train_fraction": 0.6,\n    "calibration_fraction": 0.2,\n    "reserve_fraction": 0.2,\n    "partition_rule": "Lexicographic combinations_with_replacement, PCG64 permutation; floor(.6*N) train, floor(.2*N) calibration, remainder reserved.",\n    "weighting": "Uniform ordered examples within each partition; family assignment, not independent permutation assignment.",\n    "reserve_policy": "Reserve family IDs are assigned but their ordered examples, labels and model outputs are not constructed or evaluated. IDs are a procedural holdout, not a secrecy or cryptographic guarantee.",\n    "claim_boundary": "Unseen canonical operand combinations within the same modulus and length; not length extrapolation or proof of a general algorithm."\n  },\n  "model": {\n    "source": "Unchanged T1_source_0.1.0 tiny_autograd.py and tiny_transformer.py; a new weight-tied wrapper reuses TransformerBlock and readout.",\n    "d_model": 24,\n    "n_heads": 2,\n    "d_ff": 48,\n    "n_steps": 6,\n    "shared_block_count": 1,\n    "layer_norm_epsilon": 0.00001,\n    "dropout": 0.0,\n    "dtype": "float32",\n    "readout": "Shared CLS LayerNorm plus 17-class linear head, trained on final step only. Intermediate head alignment is diagnostic, not guaranteed.",\n    "seed": 2003\n  },\n  "training": {\n    "updates": 1200,\n    "batch_size": 64,\n    "sampling": "With replacement from training ordered examples only, independent PCG64 stream.",\n    "sampling_seed": 314159,\n    "optimizer": "Original NumPy Adam",\n    "learning_rate": 0.001,\n    "beta1": 0.9,\n    "beta2": 0.999,\n    "epsilon": 0.00000001,\n    "max_grad_norm": 1.0,\n    "weight_decay": 0.0,\n    "loss": "Final-step multiclass cross-entropy only",\n    "auxiliary_loss_weight": 0.0,\n    "checkpoints": [0, 100, 400, 1200],\n    "checkpoint_selection": "None: retain and report every scheduled checkpoint; final readiness uses update 1200 only.",\n    "progress_every": 100,\n    "evaluation_batch_size": 64,\n    "early_stop": "Non-finite loss/gradient/weights or method assertion failure only; record incomplete and do not change scientific settings."\n  },\n  "measurements": {\n    "primary_future_observer_target": "Z_l = 1[argmax(logits_l) != argmax(logits_6)]",\n    "eligible_candidate_steps": [2, 3, 4, 5],\n    "target_is_not": "Correctness, consciousness, or the value of continuing computation",\n    "competence_metric": "Final-step exact answer accuracy on all calibration ordered examples",\n    "competence_gate": 0.9,\n    "competence_gate_rationale": "Development readiness requires at least nine correct answers in ten; this is a declared screening target, not deployment validation.",\n    "target_gate_min_events_per_class": 50,\n    "target_gate_min_families_per_class": 25,\n    "target_gate_min_class_fraction": 0.1,\n    "target_gate_min_eligible_steps": 2,\n    "target_gate_rationale": "Avoid a nearly constant binary target and apparent replication from permutations; not a power calculation for the future observer study.",\n    "record": ["accuracy_at_every_step", "decision_flip_counts", "flip_prevalence", "families_with_each_flip_class", "full_state_resume_max_error", "training_loss", "gradient_norm", "parameter_count", "elapsed_time"]\n  },\n  "observables_for_later_stage": {\n    "limited_current": "Current 24-D CLS plus 17 probabilities, natural-log entropy and top-two probability margin = 43 dimensions",\n    "all_token_current": "All 4x24 residuals plus the same prediction summaries = 115 dimensions",\n    "history": "Append past-step CLS+prediction summaries from steps 1 through l-1 to the identical current prefix; omit step 0",\n    "forbidden_inputs": ["true_label", "current_true_label_loss", "future_logits", "future_state", "future_correctness"],\n    "attention": "Not included. Any future attention-history analysis must also include current-attention controls.",\n    "required_future_controls": ["strong_nonlinear_current_only", "dimension_and_compute_accounting", "sample_mismatched_history", "train_only_residualized_history", "per_example_past_permutation_with_current_fixed"],\n    "fitting_and_decision_rules": "Not executable in P1-A0. Freeze families, parameter counts, tuning/update/inference budgets, scaling, Brier-score margin, absolute accuracy/skill criterion, calibration sensitivity, family-level intervals, seed aggregation and confirmation rules before any observer fit."\n  },\n  "outcome_rules": {\n    "methods_failed": "Invalid implementation; no scientific interpretation.",\n    "competence_failed": "This recipe did not reach the declared competence gate; no support for or against present-state sufficiency.",\n    "target_failed": "Competence may be adequate but the decision-change target is insufficiently varied; no sufficiency conclusion.",\n    "ready_for_observer_design": "Methods, final competence and target gates pass; permits design of a separate observer study, not automatic fitting or confirmation.",\n    "practice_claim": "No claim that practice reduces the history advantage can be made without observer comparisons at fixed checkpoints and non-degenerate, comparable evaluation conditions."\n  },\n  "sources": [\n    "https://drive.google.com/file/d/1116fOMjOUzIfyT-oYoI9gKv1bfoX63g9/view",\n    "https://drive.google.com/file/d/1DyJAbpnvQXhIr_pqDwGJvQTU13VfypXf/view",\n    "https://drive.google.com/file/d/1MwJIAc5stQ-6_987_ZPyBaa7L5M4ruLD/view",\n    "https://arxiv.org/abs/2201.02177"\n  ]\n}\n', 'prior/P1_A0_Report.md': '# P1-A0 — Arithmetic readiness pilot\n\nResearch 002 · 15 September 2026 · protocol v1.0.0  \n**Outcome: competence gate failed; no history predictors fitted.**\n\nThe new present-state arithmetic branch has started. The implementation checks\npassed, but this one fixed training recipe did not learn to answer unseen operand\ncombinations reliably. The prespecified stop was respected. The present-state\nsufficiency and practice hypotheses remain **untested**, not refuted or supported.\n\n## What actually ran\n\n- Task: `(a+b+c) mod 17`, with inputs `[CLS, a, b, c]`.\n- One newly initialized model, seed 2003; no prior models or reserved seeds used.\n- Original T1 NumPy Transformer block reused for six steps; width 24, two heads,\n  feed-forward width 48; 5,873 trainable parameters.\n- Exactly 1,200 Adam updates, batch size 64, learning rate 0.001, final-only loss.\n- 2,992 training examples from 581 canonical families; 950 calibration examples\n  from 193 different families. All operand permutations share a partition.\n- 195 additional families reserved; no reserved examples or outputs evaluated.\n- Local execution took 25.4 seconds, including scheduled\n  evaluation and saving; Colab runtime may differ. No interactive Colab session\n  was executed here. The delivered notebook is syntax-checked and uses the same\n  locally executed code with NumPy only.\n\n## Learning across the fixed checkpoints\n\n| Training updates | Training accuracy | Unseen-combination accuracy |\n|---:|---:|---:|\n| 0 | 4.91% | 4.84% |\n| 100 | 8.92% | 0.95% |\n| 400 | 20.35% | 0.84% |\n| 1,200 | 68.11% | 3.16% |\n\nThe final calibration result was 30 correct answers out of 950, versus the\ndeclared 90% development competence target. Uniform random guessing among 17\nclasses has expected accuracy 5.88%; this is a mathematical reference, not a\nfitted baseline or an independent significance test. Permutations are correlated.\n\nThe train/calibration gap is **consistent with fitting training combinations\nwithout learning a reliably generalizing arithmetic rule**. This single short\nrun does not establish that the architecture cannot learn modular arithmetic,\nthat longer training would succeed, or that its failures arise from one specific\nmechanism. No post-outcome extension or hyperparameter search was run.\n\n## Decision-change target at update 1,200\n\n| Computation step | Changed / total | Flip rate | Distinct families: changed / unchanged | Eligible |\n|---:|---:|---:|---:|---|\n| 2 | 798 / 950 | 84.00% | 170 / 38 | Yes |\n| 3 | 654 / 950 | 68.84% | 145 / 76 | Yes |\n| 4 | 475 / 950 | 50.00% | 121 / 118 | Yes |\n| 5 | 244 / 950 | 25.68% | 77 / 170 | Yes |\n\nAll four candidate steps had enough examples and distinct families in both\nclasses under the declared anti-degeneracy screen. A family can contribute to\nboth classes through different permutations; the two family counts are not\ndisjoint sample sizes. The target gate passed, but competence did not, so readiness\nfailed. Changes in an unreliable network are not evidence of useful self-monitoring.\n\n## Mathematical and implementation checks\n\nEvery method check passed: exact labels, deterministic family assignment, split\nseparation, unique weight-tied parameters, gradients checked by finite differences,\nprobability normalization, checkpoint round trips, and observer interfaces that\ndo not use labels or future states.\n\nAt all four saved training checkpoints, resuming from any of steps 0..6 reproduced\nthe final logits for the 16 audited calibration examples with maximum absolute\ndifference **0.0** in this runtime. This checks exact computational continuation\nfrom the complete current state. It costs the remaining Transformer computation\nand does not show that an inexpensive current-only observer predicts equally well.\nIt is expected for this deterministic architecture, even before training.\n\nFinite binary truth-table controls gave oracle Brier losses 0.25 for an incomplete\ncurrent view and 0.0 when the missing bit was revealed; a complete current view\nalready had oracle loss 0.0. These are mathematical fixtures, not fitted-probe\npower estimates and not natural Transformer-history findings.\n\nThe separate verification script recomputed all calibration accuracy/flip metrics\nfrom saved arrays, all checkpoint training accuracies from saved weights, source\nidentity against the original ZIP, all 1,200 update records, and the stopping\ndecision. The prospective local protocol/source freeze remained unchanged.\n\n## Scientific boundary and next step\n\nP1-A0 is complete with status `competence_failed`. No history advantage, practical\nsufficiency, training-induced sufficiency, consciousness or causal metacognition\nclaim has been established. Earlier T1/T2 findings remain unchanged.\n\nThe next step is a separately versioned, bounded **arithmetic competence design**,\nbefore observer fitting. It must justify any task/training change, record that\nthis development outcome informed the choice, and set a finite budget and fresh\nevaluation plan before fitting. The old reserved seeds remain sealed. This report\ndoes not automatically authorize additional runs or use of held-out families.\n\n## Reproduction and files\n\n`P1_A0_Colab.ipynb` is standalone: upload it into Colab. It contains the source,\nprotocol and recorded report, runs method checks, and defaults to **not** rerunning\ntraining. Set `RUN_REPRODUCTION = True` only to reproduce the same exposed pilot;\nit is not an independent replication and may vary slightly across numerical\nlibrary versions. No GPU or external model download is needed.\n\n`P1_A0_Study.zip` contains source, original source archive, the exact protocol,\nfour trained checkpoints, calibration trajectories, development examples, optimizer\nmoments, logs, local freeze, verification and this report. Existing run directories\nare never overwritten by the experiment runner.\n\n### Sources and provenance\n\n- [Original T1 source](https://drive.google.com/file/d/1116fOMjOUzIfyT-oYoI9gKv1bfoX63g9/view).\n- [T2-E2 report](https://drive.google.com/file/d/1DyJAbpnvQXhIr_pqDwGJvQTU13VfypXf/view).\n- [Observer design constraints](https://drive.google.com/file/d/1MwJIAc5stQ-6_987_ZPyBaa7L5M4ruLD/view).\n- [Grokking study](https://arxiv.org/abs/2201.02177): background on algorithmic\n  learning/generalization, not evidence for our new hypothesis.\n', 'prior/test_methods.py': '"""Executable method audits; no natural model optimization or probe fits."""\nimport itertools\nfrom pathlib import Path\nimport tempfile\n\nimport numpy as np\n\nimport experiment as ex\nfrom src.tiny_autograd import Tensor, cross_entropy, parameter, Adam\n\n\ndef run_checks():\n    checks = {}\n    protocol = ex.read_protocol()\n    data, manifest = ex.make_partitions(protocol)\n    ids = {name: set(values) for name, values in manifest["family_ids"].items()}\n    assert len(set.union(*ids.values())) == 969\n    assert all(not ids[a] & ids[b] for a, b in itertools.combinations(ids, 2))\n    assert set(data) == {"train", "calibration"} and not manifest["reserve_examples_constructed"]\n    for split, values in data.items():\n        tokens, labels, fids = values["tokens"], values["labels"], values["family_ids"]\n        assert np.array_equal(labels, tokens[:, 1:].sum(axis=1) % 17)\n        assert np.all(tokens[:, 0] == 17)\n        assert len(np.unique(tokens, axis=0)) == len(tokens)\n        assert set(fids.tolist()) == ids[split]\n        for family in np.unique(fids):\n            rows = tokens[fids == family, 1:]\n            assert np.all(np.sort(rows, axis=1) == np.sort(rows[0]))\n    again, again_manifest = ex.make_partitions(protocol)\n    assert again_manifest == manifest\n    assert all(np.array_equal(data[s][k], again[s][k]) for s in data for k in data[s])\n    checks["exact_labels_family_separation_and_repeatability"] = True\n    checks["reserved_examples_not_constructed"] = True\n\n    model = ex.ArithmeticTransformer(protocol, seed=8001)\n    params = model.parameters()\n    assert len(params) == len(set(map(id, params)))\n    assert len(params) == len({p.name for p in params}) and len(model.layers) == 1\n    tokens = data["train"]["tokens"][:4]\n    labels = data["train"]["labels"][:4]\n    trajectory = ex.extract(model, tokens)\n    assert trajectory["states"].shape == (4, 7, 4, 24)\n    assert trajectory["probabilities"].shape == (4, 7, 17)\n    np.testing.assert_allclose(trajectory["probabilities"].sum(-1), 1, atol=2e-6)\n    checks["one_shared_block_unique_parameters_and_shapes"] = True\n    checks["normalization"] = True\n    checks["state_resume_max_errors"] = ex.resume_audit(model, trajectory, count=4)\n    try:\n        with ex.inference(model):\n            model.resume(trajectory["states"][:, 2, 0], 2)\n        raise AssertionError("CLS-only resume should be rejected.")\n    except ValueError:\n        pass\n    checks["partial_state_resume_rejected"] = True\n\n    for view, width in (("cls", 43), ("all_token", 115)):\n        for step in (2, 3, 4, 5):\n            current = ex.observable_features(trajectory, step, view, history=False)\n            history = ex.observable_features(trajectory, step, view, history=True)\n            assert current.shape[1] == width\n            assert history.shape[1] == width + (step - 1) * 43\n            np.testing.assert_array_equal(current, history[:, :width])\n            changed = {key: value.copy() for key, value in trajectory.items()}\n            changed["states"][:, step+1:] = 12345\n            changed["probabilities"][:, step+1:] = -12345\n            changed["logits"][:] = 0  # logits are not used by observer extraction\n            changed["labels"] = np.arange(4)  # labels are not used either\n            np.testing.assert_array_equal(history, ex.observable_features(changed, step, view, history=True))\n    checks["current_prefix_identical_no_future_or_label_leakage"] = True\n\n    # Finite differences audit gradients through the block reused SIX times.\n    _, logits, _ = model(tokens)\n    loss = cross_entropy(logits[-1], labels)\n    for p in params:\n        p.zero_grad()\n    loss.backward()\n    assert all(np.all(np.isfinite(p.grad)) for p in params)\n    errors = []\n    for name in ("layers.0.q_proj.weight", "layers.0.ff1.weight", "readout.weight"):\n        p = next(p for p in params if p.name == name)\n        index = np.unravel_index(np.argmax(np.abs(p.grad)), p.shape)\n        analytic = float(p.grad[index])\n        original = float(p.data[index])\n        epsilon = 1e-3\n        values = []\n        for sign in (1, -1):\n            p.data[index] = original + sign * epsilon\n            with ex.inference(model):\n                _, candidate_logits, _ = model(tokens)\n                values.append(float(cross_entropy(candidate_logits[-1], labels).data))\n        p.data[index] = original\n        numeric = (values[0] - values[1]) / (2 * epsilon)\n        assert np.isclose(analytic, numeric, rtol=0.05, atol=0.006), (name, analytic, numeric)\n        errors.append({"parameter": name, "index": [int(i) for i in index],\n                       "analytic": analytic, "finite_difference": numeric})\n    checks["tied_gradient_finite_differences"] = errors\n\n    with tempfile.TemporaryDirectory(prefix="p1_method_") as temp:\n        checkpoint = Path(temp) / "model.npz"\n        model.save(checkpoint)\n        loaded = ex.ArithmeticTransformer(protocol, seed=8002)\n        loaded.load(checkpoint)\n        a, b = ex.extract(model, tokens), ex.extract(loaded, tokens)\n        np.testing.assert_array_equal(a["logits"], b["logits"])\n    checks["serialization_exact_roundtrip"] = True\n\n    # Only scalar synthetic optimizer fixture; not natural model training.\n    scalar = parameter(np.array([2.0]), "fixture")\n    optimizer = Adam([scalar], learning_rate=0.1)\n    before = float(scalar.data[0] ** 2)\n    for _ in range(10):\n        optimizer.zero_grad()\n        objective = (scalar * scalar).sum()\n        objective.backward()\n        optimizer.step()\n    assert float(scalar.data[0] ** 2) < before\n    checks["synthetic_scalar_optimizer"] = True\n\n    # Exact oracle fixtures: current observation u vs full state (u,v).\n    truth = np.array(list(itertools.product([0, 1], repeat=2)))\n    y = truth[:, 0] ^ truth[:, 1]\n    partial = np.array([y[truth[:, 0] == u].mean() for u in truth[:, 0]])\n    full = y.astype(float)\n    partial_brier = float(np.mean((partial - y) ** 2))\n    full_brier = float(np.mean((full - y) ** 2))\n    assert partial_brier == 0.25 and full_brier == 0.0\n    checks["exact_oracle_controls"] = {\n        "partial_current_brier": partial_brier,\n        "partial_current_plus_missing_history_brier": full_brier,\n        "complete_current_brier": full_brier,\n        "complete_current_plus_redundant_history_brier": full_brier,\n        "scope": "Truth-table arithmetic, not fitted-probe power or Transformer evidence."}\n    checks["passed"] = True\n    return checks\n\n\nif __name__ == "__main__":\n    import json\n    print(json.dumps(run_checks(), indent=2))\n', 'prior/P1_A0_Protocol.md': "# P1-A0 — Present-state sufficiency after arithmetic practice\n\nResearch 002 · protocol v1.0.0 · 15 September 2026  \nStatus: a new, bounded development pilot, not a confirmatory history study.\n\n## Purpose and boundary\n\nBegin the approved arithmetic branch while retaining all negative T1/T2 findings.\nT2-E2 already used all-token current observations; repeating that comparison on its\nexposed panel is not authorized here. Its remaining true-label-loss target is\ndistinct from this branch's proposed decision-change target. Original reserved\nseeds 1103, 1201 and 1301 are not used. No previous trained weights are loaded.\n\nThe new hypothesis concerns **usable prediction under a finite observer budget**.\nIt does not operationalize subjective awareness, and no arithmetic accuracy or\nprobe result can by itself establish consciousness, causal metacognition, or a\nbenefit from an early-exit controller.\n\n## Mathematical result before experimentation\n\nFor fixed weights theta, a fixed total step count L, no stochasticity, no incoming\nnew inputs, and complete all-token residual state S_l at a block boundary:\n\n`S_(l+1) = F_theta(S_l)` and `logits_L = readout(F_theta^(L-l)(S_l))`.\n\nInduction gives exact functional dependence on the current state and step. Thus\nearlier activation records are unnecessary to **resume the actual computation**.\nThis holds before training as well as after it. Our numerical resume check is an\nimplementation audit of that property, not a discovery or a test of cheap\npredictability. Resumption pays the cost of all remaining Transformer steps.\n\nThe complete state can encode the past. It is not the latest input token alone,\nand it does not mean removing learned weights or previous operands. For this\nencoder wrapper there are no KV caches, dropout, external inputs, or recurrent\nvariables outside the saved residual tensor; the weights and step remain fixed.\nThe statement determines the model's future output, not whether that output is\nmathematically correct. True-label losses also require the true label.\n\nThe nontrivial future experiment asks whether an **affordable** current-only\nobserver predicts future decision change accurately, with history's possible\nincrement smaller than a justified, prospectively fixed margin. Equal performance\nfrom two poor observers and a nonsignificant difference are insufficient evidence.\nThe further practice hypothesis asks whether that increment declines across\npredeclared training checkpoints. This first stage does not fit those observers.\n\n## Why this task and implementation\n\nUse `(a+b+c) mod 17` with three operands in 0..16. Labels are exact and inexpensive;\nthere are 4,913 ordered triples and 969 canonical sorted-triple families. All\npermutations of a triple stay in one partition. A randomly assigned 60% family\ntraining set is separated from a 20% development calibration set and the remaining\nreserved families. The reserved families are assigned IDs but not expanded into\nexamples or evaluated in this stage. This tests unseen operand combinations,\nnot new lengths, new moduli, or universal arithmetic reasoning.\n\nThe implementation reuses the original T1 NumPy autodiff and pre-norm Transformer\nblock unchanged, wrapped as one block repeated six times. It has width 24, two\nheads, feed-forward width 48 and a shared 17-class CLS readout. The readout is\ntrained at the final step only; intermediate readout alignment is not assumed.\nThis is a new arithmetic regime, not a claim of numerical identity with U7C.\n\nOne development model, seed 2003, receives exactly 1,200 Adam updates at learning\nrate 0.001, batch size 64, global gradient clip 1.0, no weight decay and no\nintermediate supervision. Checkpoints are 0, 100, 400 and 1,200 updates. No best\ncheckpoint is selected. The finite budget limits this turn's exploration; failure\nto learn within it is not a theorem about the task or architecture.\n\nThe canonical machine-readable specification is `P1_A0_protocol.json`. A local\n`FREEZE.json` binds the protocol, source and original source archive before the\npilot's first training update. Hashes provide a local audit trail, not an\nindependently timestamped external preregistration. Reproduction is explicitly a\nrerun of the same exposed development study, never a fresh independent replicate.\n\n## Readiness measurements and fixed stopping rules\n\nAt every checkpoint, report training and calibration accuracy for steps 0..6.\nAt candidate steps 2..5, record\n`Z_l = 1[current argmax differs from final argmax]`, which measures change rather\nthan correctness. Report both event classes and the number of distinct families\ncontributing to each class. Count families, not permutations, when interpreting\nindependent support. These are descriptive development measurements, not p-values.\n\nFinal readiness requires:\n\n- All method tests and numeric checks pass.\n- Update-1,200 final calibration accuracy is at least 90%: a declared development\n  target, not a validated operational threshold.\n- At least two of steps 2..5 have at least 50 examples **and** 25 families in each\n  flip class, with each class at least 10% of examples. These anti-degeneracy rules\n  are not a power analysis.\n\nStop after the fixed schedule even if competence fails. A competence failure\nmeans this training recipe is not ready; a target failure means this target is\nnot ready. Neither supports nor refutes present-state sufficiency. A pass permits\nthe next **design stage**, not automatic probes or confirmation. Numeric failure\nstops the run, preserves diagnostics and invalidates scientific interpretation.\n\n## Mandatory checks\n\nCheck exact arithmetic labels; canonical-family disjointness and full coverage;\nreserved examples not instantiated; unique tied parameters; finite-difference\ngradients through the shared block; probability normalization; input and output\nshapes; serialization round trip; and future-logit reproduction from each saved\ncurrent state using the same weights and remaining step count.\n\nCheck the future observer extraction interface uses only steps at or before l,\nkeeps the current prefix identical, and is insensitive to changes to future\narrays. Two finite binary truth-table fixtures distinguish full observation\n(zero oracle Brier loss) from partial observation (oracle Brier loss 0.25, reduced\nto zero by revealing the missing bit). These are mathematical fixtures, not\nempirical Transformer evidence and not sensitivity validation of any fitted probe.\n\n## Boundary before P1-A1 observer fitting\n\nP1-A0 outputs candidate observables, not an observer implementation. Before fitting\nanything, prospectively freeze a separate protocol specifying:\n\n- The finite development/replication budget and fresh training seeds; do not count\n  multiple training checkpoints or steps as independent model replicates.\n- Observer splits keeping every canonical family and all its checkpoints/steps\n  together. No base-model selection on observer test outcomes.\n- CLS-only current (43-D) and all-token current (115-D), each compared with the\n  same current prefix plus past CLS/prediction summaries from steps 1..l-1.\n- Linear and a stronger nonlinear current-only reference; matched feature-width,\n  parameter, optimization, tuning and inference resource accounting.\n- Sample-mismatched and train-only-residualized history controls; per-example\n  permutation of past slots with the current observation fixed for chronology.\n  At least two past slots are required; a fixed column reversal is insufficient.\n- Primary proper score (proposed Brier), an absolute useful-prediction criterion,\n  an independently justified equivalence/noninferiority margin, train-only\n  eligibility rules, multiplicity/aggregation and family-aware uncertainty.\n- Null and known-positive calibration through the **actual fitted pipeline**,\n  with enough sensitivity around the chosen margin. The exact oracle fixtures\n  below cannot substitute for this check.\n- Prediction-cost justification: the original T2-E2 5% relative-MSE continuation\n  threshold is not inherited as a validated Brier-score or deployment margin.\n\nNo scientific rule may be selected using a future history-versus-current test\ngain. Changes following development results need a new version and provenance.\n\n## Sources\n\n- [Original T1 source](https://drive.google.com/file/d/1116fOMjOUzIfyT-oYoI9gKv1bfoX63g9/view).\n- [T2-E2 completed report](https://drive.google.com/file/d/1DyJAbpnvQXhIr_pqDwGJvQTU13VfypXf/view).\n- [Observer design constraints](https://drive.google.com/file/d/1MwJIAc5stQ-6_987_ZPyBaa7L5M4ruLD/view).\n- [Power et al., Grokking](https://arxiv.org/abs/2201.02177): precedent for studying\n  learning/generalization on algorithmically generated datasets. It does not\n  establish our sufficiency hypothesis or prescribe this pilot's recipe.\n", 'prior/vendor/t1_original/src/tiny_transformer.py': '"""Instrumented encoder-only Transformer for T1."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .tiny_autograd import Tensor, gelu, parameter, softmax\n\n\n@dataclass(frozen=True)\nclass ModelConfig:\n    vocab_size: int\n    sequence_length: int\n    d_model: int\n    n_heads: int\n    d_ff: int\n    n_layers: int\n    n_classes: int = 2\n    layer_norm_epsilon: float = 1e-5\n\n\nclass Module:\n    def parameters(self) -> list[Tensor]:\n        result: list[Tensor] = []\n        for value in self.__dict__.values():\n            if isinstance(value, Tensor) and value.requires_grad:\n                result.append(value)\n            elif isinstance(value, Module):\n                result.extend(value.parameters())\n            elif isinstance(value, (list, tuple)):\n                for item in value:\n                    if isinstance(item, Module):\n                        result.extend(item.parameters())\n                    elif isinstance(item, Tensor) and item.requires_grad:\n                        result.append(item)\n        return result\n\n\nclass Linear(Module):\n    def __init__(self, in_features: int, out_features: int, rng: np.random.Generator, name: str) -> None:\n        scale = np.sqrt(2.0 / (in_features + out_features))\n        self.weight = parameter(\n            rng.normal(0.0, scale, size=(in_features, out_features)),\n            f"{name}.weight",\n        )\n        self.bias = parameter(np.zeros(out_features), f"{name}.bias")\n\n    def __call__(self, x: Tensor) -> Tensor:\n        return x @ self.weight + self.bias\n\n\nclass LayerNorm(Module):\n    def __init__(self, features: int, epsilon: float, name: str) -> None:\n        self.weight = parameter(np.ones(features), f"{name}.weight")\n        self.bias = parameter(np.zeros(features), f"{name}.bias")\n        self.epsilon = epsilon\n\n    def __call__(self, x: Tensor) -> Tensor:\n        mean = x.mean(axis=-1, keepdims=True)\n        centered = x - mean\n        variance = (centered * centered).mean(axis=-1, keepdims=True)\n        normalized = centered / ((variance + self.epsilon) ** 0.5)\n        return normalized * self.weight + self.bias\n\n\nclass TransformerBlock(Module):\n    def __init__(self, config: ModelConfig, rng: np.random.Generator, index: int) -> None:\n        if config.d_model % config.n_heads:\n            raise ValueError("d_model must be divisible by n_heads")\n        prefix = f"layers.{index}"\n        self.config = config\n        self.norm1 = LayerNorm(config.d_model, config.layer_norm_epsilon, f"{prefix}.norm1")\n        self.q_proj = Linear(config.d_model, config.d_model, rng, f"{prefix}.q_proj")\n        self.k_proj = Linear(config.d_model, config.d_model, rng, f"{prefix}.k_proj")\n        self.v_proj = Linear(config.d_model, config.d_model, rng, f"{prefix}.v_proj")\n        self.o_proj = Linear(config.d_model, config.d_model, rng, f"{prefix}.o_proj")\n        self.norm2 = LayerNorm(config.d_model, config.layer_norm_epsilon, f"{prefix}.norm2")\n        self.ff1 = Linear(config.d_model, config.d_ff, rng, f"{prefix}.ff1")\n        self.ff2 = Linear(config.d_ff, config.d_model, rng, f"{prefix}.ff2")\n\n    def __call__(self, x: Tensor) -> tuple[Tensor, Tensor]:\n        batch, length, width = x.shape\n        heads = self.config.n_heads\n        head_width = width // heads\n\n        normalized = self.norm1(x)\n        q = self.q_proj(normalized).reshape(batch, length, heads, head_width).transpose(0, 2, 1, 3)\n        k = self.k_proj(normalized).reshape(batch, length, heads, head_width).transpose(0, 2, 1, 3)\n        v = self.v_proj(normalized).reshape(batch, length, heads, head_width).transpose(0, 2, 1, 3)\n        scores = (q @ k.transpose(0, 1, 3, 2)) / np.sqrt(float(head_width))\n        attention = softmax(scores, axis=-1)\n        attended = attention @ v\n        attended = attended.transpose(0, 2, 1, 3).reshape(batch, length, width)\n        x = x + self.o_proj(attended)\n        x = x + self.ff2(gelu(self.ff1(self.norm2(x))))\n        return x, attention\n\n\nclass TinyTransformer(Module):\n    """A standard stack with a shared layer-wise readout for instrumentation."""\n\n    def __init__(self, config: ModelConfig, seed: int) -> None:\n        self.config = config\n        rng = np.random.default_rng(seed)\n        self.token_embedding = parameter(\n            rng.normal(0.0, 0.02, size=(config.vocab_size, config.d_model)),\n            "token_embedding",\n        )\n        self.position_embedding = parameter(\n            rng.normal(0.0, 0.02, size=(config.sequence_length, config.d_model)),\n            "position_embedding",\n        )\n        self.layers = [TransformerBlock(config, rng, i) for i in range(config.n_layers)]\n        self.readout_norm = LayerNorm(config.d_model, config.layer_norm_epsilon, "readout_norm")\n        self.readout = Linear(config.d_model, config.n_classes, rng, "readout")\n\n    def _read(self, state: Tensor) -> Tensor:\n        return self.readout(self.readout_norm(state[:, 0, :]))\n\n    def __call__(\n        self,\n        tokens: np.ndarray,\n    ) -> tuple[list[Tensor], list[Tensor], list[Tensor]]:\n        batch, length = tokens.shape\n        if length != self.config.sequence_length:\n            raise ValueError(f"expected sequence length {self.config.sequence_length}, got {length}")\n        one_hot = np.eye(self.config.vocab_size, dtype=np.float32)[tokens]\n        state = Tensor(one_hot) @ self.token_embedding + self.position_embedding\n        states = [state]\n        logits = [self._read(state)]\n        attentions: list[Tensor] = []\n        for layer in self.layers:\n            state, attention = layer(state)\n            states.append(state)\n            logits.append(self._read(state))\n            attentions.append(attention)\n        return states, logits, attentions\n\n    def state_dict(self) -> dict[str, np.ndarray]:\n        return {\n            parameter_tensor.name or f"parameter_{index}": parameter_tensor.data.copy()\n            for index, parameter_tensor in enumerate(self.parameters())\n        }\n\n    def save(self, path: Path) -> None:\n        np.savez_compressed(path, **self.state_dict())\n\n\ndef numpy_softmax(logits: np.ndarray) -> np.ndarray:\n    shifted = logits - logits.max(axis=-1, keepdims=True)\n    values = np.exp(shifted)\n    return values / values.sum(axis=-1, keepdims=True)\n', 'prior/vendor/t1_original/src/__init__.py': '"""T1 experiment package."""\n\n', 'prior/vendor/t1_original/src/tiny_autograd.py': '"""A small NumPy reverse-mode autodiff engine for the T1 experiment.\n\nIt intentionally implements only the tensor operations used by the tiny\nTransformer.  The experiment therefore has no deep-learning-framework\ndependency and remains portable to a constrained CPU environment.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Iterable, Sequence\n\nimport numpy as np\n\n\ndef _as_float_array(value: object) -> np.ndarray:\n    return np.asarray(value, dtype=np.float32)\n\n\ndef _unbroadcast(grad: np.ndarray, shape: tuple[int, ...]) -> np.ndarray:\n    """Sum a broadcast gradient back to an operand\'s original shape."""\n    while grad.ndim > len(shape):\n        grad = grad.sum(axis=0)\n    for axis, size in enumerate(shape):\n        if size == 1 and grad.shape[axis] != 1:\n            grad = grad.sum(axis=axis, keepdims=True)\n    return grad.reshape(shape)\n\n\nclass Tensor:\n    """A differentiable NumPy array."""\n\n    def __init__(\n        self,\n        data: object,\n        *,\n        requires_grad: bool = False,\n        name: str | None = None,\n        _children: Sequence["Tensor"] = (),\n    ) -> None:\n        self.data = _as_float_array(data)\n        self.requires_grad = requires_grad\n        self.grad = np.zeros_like(self.data) if requires_grad else None\n        self.name = name\n        self._prev = tuple(child for child in _children if child.requires_grad)\n        self._backward = lambda: None\n\n    @property\n    def shape(self) -> tuple[int, ...]:\n        return self.data.shape\n\n    @property\n    def ndim(self) -> int:\n        return self.data.ndim\n\n    @staticmethod\n    def ensure(value: object) -> "Tensor":\n        return value if isinstance(value, Tensor) else Tensor(value)\n\n    def zero_grad(self) -> None:\n        if self.requires_grad:\n            self.grad.fill(0.0)\n\n    def backward(self) -> None:\n        if self.data.size != 1:\n            raise ValueError("backward() requires a scalar Tensor")\n        topo: list[Tensor] = []\n        visited: set[int] = set()\n\n        def build(node: Tensor) -> None:\n            node_id = id(node)\n            if node_id in visited:\n                return\n            visited.add(node_id)\n            for child in node._prev:\n                build(child)\n            topo.append(node)\n\n        build(self)\n        self.grad = np.ones_like(self.data)\n        for node in reversed(topo):\n            node._backward()\n\n    def __add__(self, other: object) -> "Tensor":\n        other = Tensor.ensure(other)\n        out = Tensor(\n            self.data + other.data,\n            requires_grad=self.requires_grad or other.requires_grad,\n            _children=(self, other),\n        )\n\n        def backward() -> None:\n            if self.requires_grad:\n                self.grad += _unbroadcast(out.grad, self.shape)\n            if other.requires_grad:\n                other.grad += _unbroadcast(out.grad, other.shape)\n\n        out._backward = backward\n        return out\n\n    def __radd__(self, other: object) -> "Tensor":\n        return self + other\n\n    def __neg__(self) -> "Tensor":\n        return self * -1.0\n\n    def __sub__(self, other: object) -> "Tensor":\n        return self + (-Tensor.ensure(other))\n\n    def __rsub__(self, other: object) -> "Tensor":\n        return Tensor.ensure(other) - self\n\n    def __mul__(self, other: object) -> "Tensor":\n        other = Tensor.ensure(other)\n        out = Tensor(\n            self.data * other.data,\n            requires_grad=self.requires_grad or other.requires_grad,\n            _children=(self, other),\n        )\n\n        def backward() -> None:\n            if self.requires_grad:\n                self.grad += _unbroadcast(out.grad * other.data, self.shape)\n            if other.requires_grad:\n                other.grad += _unbroadcast(out.grad * self.data, other.shape)\n\n        out._backward = backward\n        return out\n\n    def __rmul__(self, other: object) -> "Tensor":\n        return self * other\n\n    def __truediv__(self, other: object) -> "Tensor":\n        return self * (Tensor.ensure(other) ** -1.0)\n\n    def __rtruediv__(self, other: object) -> "Tensor":\n        return Tensor.ensure(other) / self\n\n    def __pow__(self, exponent: float) -> "Tensor":\n        out = Tensor(\n            self.data**exponent,\n            requires_grad=self.requires_grad,\n            _children=(self,),\n        )\n\n        def backward() -> None:\n            if self.requires_grad:\n                self.grad += out.grad * exponent * (self.data ** (exponent - 1.0))\n\n        out._backward = backward\n        return out\n\n    def __matmul__(self, other: object) -> "Tensor":\n        other = Tensor.ensure(other)\n        out = Tensor(\n            self.data @ other.data,\n            requires_grad=self.requires_grad or other.requires_grad,\n            _children=(self, other),\n        )\n\n        def backward() -> None:\n            if self.requires_grad:\n                grad = out.grad @ np.swapaxes(other.data, -1, -2)\n                self.grad += _unbroadcast(grad, self.shape)\n            if other.requires_grad:\n                grad = np.swapaxes(self.data, -1, -2) @ out.grad\n                other.grad += _unbroadcast(grad, other.shape)\n\n        out._backward = backward\n        return out\n\n    def sum(\n        self,\n        axis: int | tuple[int, ...] | None = None,\n        keepdims: bool = False,\n    ) -> "Tensor":\n        out = Tensor(\n            self.data.sum(axis=axis, keepdims=keepdims),\n            requires_grad=self.requires_grad,\n            _children=(self,),\n        )\n\n        def backward() -> None:\n            if not self.requires_grad:\n                return\n            grad = out.grad\n            if axis is not None and not keepdims:\n                axes = (axis,) if isinstance(axis, int) else axis\n                axes = tuple(a if a >= 0 else a + self.ndim for a in axes)\n                for current_axis in sorted(axes):\n                    grad = np.expand_dims(grad, current_axis)\n            self.grad += np.broadcast_to(grad, self.shape)\n\n        out._backward = backward\n        return out\n\n    def mean(\n        self,\n        axis: int | tuple[int, ...] | None = None,\n        keepdims: bool = False,\n    ) -> "Tensor":\n        if axis is None:\n            divisor = self.data.size\n        else:\n            axes = (axis,) if isinstance(axis, int) else axis\n            divisor = int(np.prod([self.shape[a] for a in axes]))\n        return self.sum(axis=axis, keepdims=keepdims) / float(divisor)\n\n    def reshape(self, *shape: int) -> "Tensor":\n        out = Tensor(\n            self.data.reshape(*shape),\n            requires_grad=self.requires_grad,\n            _children=(self,),\n        )\n\n        def backward() -> None:\n            if self.requires_grad:\n                self.grad += out.grad.reshape(self.shape)\n\n        out._backward = backward\n        return out\n\n    def transpose(self, *axes: int) -> "Tensor":\n        out = Tensor(\n            self.data.transpose(*axes),\n            requires_grad=self.requires_grad,\n            _children=(self,),\n        )\n        inverse = np.argsort(axes)\n\n        def backward() -> None:\n            if self.requires_grad:\n                self.grad += out.grad.transpose(*inverse)\n\n        out._backward = backward\n        return out\n\n    def __getitem__(self, index: object) -> "Tensor":\n        out = Tensor(\n            self.data[index],\n            requires_grad=self.requires_grad,\n            _children=(self,),\n        )\n\n        def backward() -> None:\n            if self.requires_grad:\n                np.add.at(self.grad, index, out.grad)\n\n        out._backward = backward\n        return out\n\n    def exp(self) -> "Tensor":\n        values = np.exp(self.data)\n        out = Tensor(values, requires_grad=self.requires_grad, _children=(self,))\n\n        def backward() -> None:\n            if self.requires_grad:\n                self.grad += out.grad * values\n\n        out._backward = backward\n        return out\n\n    def log(self) -> "Tensor":\n        out = Tensor(\n            np.log(self.data),\n            requires_grad=self.requires_grad,\n            _children=(self,),\n        )\n\n        def backward() -> None:\n            if self.requires_grad:\n                self.grad += out.grad / self.data\n\n        out._backward = backward\n        return out\n\n    def tanh(self) -> "Tensor":\n        values = np.tanh(self.data)\n        out = Tensor(values, requires_grad=self.requires_grad, _children=(self,))\n\n        def backward() -> None:\n            if self.requires_grad:\n                self.grad += out.grad * (1.0 - values**2)\n\n        out._backward = backward\n        return out\n\n\ndef parameter(data: object, name: str) -> Tensor:\n    return Tensor(data, requires_grad=True, name=name)\n\n\ndef softmax(x: Tensor, axis: int = -1) -> Tensor:\n    stable = x - np.max(x.data, axis=axis, keepdims=True)\n    exp_values = stable.exp()\n    return exp_values / exp_values.sum(axis=axis, keepdims=True)\n\n\ndef gelu(x: Tensor) -> Tensor:\n    coefficient = np.sqrt(2.0 / np.pi)\n    return 0.5 * x * (1.0 + (coefficient * (x + 0.044715 * (x**3.0))).tanh())\n\n\ndef cross_entropy(logits: Tensor, labels: np.ndarray) -> Tensor:\n    if logits.ndim != 2:\n        raise ValueError("cross_entropy expects logits with shape [batch, classes]")\n    one_hot = np.eye(logits.shape[1], dtype=np.float32)[labels]\n    shifted = logits - np.max(logits.data, axis=1, keepdims=True)\n    log_prob = shifted - shifted.exp().sum(axis=1, keepdims=True).log()\n    return -(log_prob * one_hot).sum(axis=1).mean()\n\n\nclass Adam:\n    """Minimal Adam with global gradient clipping."""\n\n    def __init__(\n        self,\n        parameters: Iterable[Tensor],\n        *,\n        learning_rate: float,\n        beta1: float = 0.9,\n        beta2: float = 0.999,\n        epsilon: float = 1e-8,\n        max_grad_norm: float = 1.0,\n    ) -> None:\n        unique: dict[int, Tensor] = {id(p): p for p in parameters}\n        self.parameters = list(unique.values())\n        self.learning_rate = learning_rate\n        self.beta1 = beta1\n        self.beta2 = beta2\n        self.epsilon = epsilon\n        self.max_grad_norm = max_grad_norm\n        self.m = [np.zeros_like(p.data) for p in self.parameters]\n        self.v = [np.zeros_like(p.data) for p in self.parameters]\n        self.step_count = 0\n\n    def zero_grad(self) -> None:\n        for p in self.parameters:\n            p.zero_grad()\n\n    def step(self) -> float:\n        self.step_count += 1\n        squared_norm = sum(float(np.sum(p.grad**2)) for p in self.parameters)\n        grad_norm = float(np.sqrt(squared_norm))\n        scale = min(1.0, self.max_grad_norm / (grad_norm + 1e-12))\n\n        beta1_correction = 1.0 - self.beta1**self.step_count\n        beta2_correction = 1.0 - self.beta2**self.step_count\n        for index, p in enumerate(self.parameters):\n            grad = p.grad * scale\n            self.m[index] = self.beta1 * self.m[index] + (1.0 - self.beta1) * grad\n            self.v[index] = self.beta2 * self.v[index] + (1.0 - self.beta2) * (grad**2)\n            m_hat = self.m[index] / beta1_correction\n            v_hat = self.v[index] / beta2_correction\n            p.data -= self.learning_rate * m_hat / (np.sqrt(v_hat) + self.epsilon)\n        return grad_norm\n'}
SOURCE_HASHES={'run_a0b.py': 'bbdd437792866de693af74b8d9aa33250c52a8137d89f65d9799919ad4f3eebf', 'checks_a0b.py': '2f9a8845ee936238562826b870ea9dafa1ca380e922d43583fc9a1c574c88204', 'verify_a0b.py': '0caab385fa3a31468065e7794cae6d7865b645b998259e0fdeef8e73be762f57', 'P1_A0B_protocol.json': 'd1aa382b2f38f1a88ad84de5f68d8adfea4ede35318a7c9e40d1b39bdd18f888', 'P1_A0B_Protocol.md': '949f9199821574c526af7c4cea9723431166d416fe0a77f9b9da02307d6167ec', 'prior/experiment.py': 'c57bc85f56178c701e77a29cf8b0328479d888c8d1869ab97171b7579fbe59f1', 'prior/P1_A0_protocol.json': '6b7f615fcfd6db87098f90330a9260eee0717025c3678b9ed3e37770a47ca482', 'prior/P1_A0_Report.md': 'd960f27285a3f42d6d6df4f7083afe68d3bd907c9d811df69748c2818d26edab', 'prior/test_methods.py': '0f844a7e3378ceffdd3b0af9bbb548253b504620d0eb94ec935847e53aebf3d6', 'prior/P1_A0_Protocol.md': '834684f53c6222d0e3edb3f05461a56a52414ffa3a8368bbc1e42d70f8a6b363', 'prior/vendor/t1_original/src/tiny_transformer.py': '3278f6a7177caa5478dac30b1912724d2f30c187a891043c547bbaf19d6521dc', 'prior/vendor/t1_original/src/__init__.py': '3a526a67d00e49be7d87d98b7f79fd57815d5f8b5524030abb5be34b5d8a9162', 'prior/vendor/t1_original/src/tiny_autograd.py': '18fb91003d171b881c7afeb909fa08c278dd3f19ef93dbd71a2e9bbd7089d61f'}
for name,content in SOURCES.items():
    assert hashlib.sha256(content.encode()).hexdigest()==SOURCE_HASHES[name]
    path=ROOT/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content)
(ROOT/'prior/vendor/T1_source_0.1.0.zip').write_bytes(base64.b64decode('...'))
RECORDED_REPORT="# P1-A0B — Arithmetic competence revision\n\nResearch 002 · 15 September 2026 · protocol v1.0.0  \n**Development candidate: matrix_decay; observer fitting still unopened.**\n\n## What was completed\n\nFour fixed runs tested 6,000 updates with and without matrix-only decoupled weight\ndecay, for new model seeds 2017 and 2027. All 24,000 updates were completed; no\nextra seeds, extensions or best-checkpoint selection were added. Initial weights\nand the entire training batch stream were identical within each seed pair.\n\nThe task, split and model were unchanged from P1-A0: `(a+b+c) mod 17`, a six-step\nweight-tied Transformer, width 24, two heads, feed-forward width 48 and 5,873 trainable\nparameters. Both arms used final-only cross entropy, batch size 64, learning rate\n0.001, Adam beta1=0.9/beta2=0.999 and gradient clipping at 1.0. Decay was either\nzero or 1 on matrices only; biases and LayerNorm vectors were excluded. At learning\nrate 0.001, decay 1 multiplies each matrix by 0.999 before the adaptive update.\n\nP1-A0's observed train/calibration gap motivated this development revision.\nLonger training and weight decay have precedent in small algorithmic studies,\nbut that literature uses different architectures and much larger budgets; it\ndoes not guarantee success in our setting. [Grokking](https://arxiv.org/abs/2201.02177),\n[decoupled weight decay](https://arxiv.org/abs/1711.05101).\n\n## Final results at the fixed 6,000-update endpoint\n\n| Seed | Regimen | Final training accuracy | Final calibration accuracy | Eligible steps | Competence >=90% |\n|---:|---|---:|---:|---|---|\n| 2017 | no_decay | 97.73% | 0.63% | 2, 3, 4, 5 | No |\n| 2017 | matrix_decay | 99.93% | 99.58% | 2, 3, 4 | Yes |\n| 2027 | no_decay | 98.46% | 1.26% | 2, 3, 4, 5 | No |\n| 2027 | matrix_decay | 98.40% | 98.00% | 3, 4 | Yes |\n\nThe paired accuracy differences, decay minus no-decay, were seed 2017: +98.95 percentage points; seed 2027: +96.74 percentage points.\nThe arithmetic mean was **+97.84 percentage points**.\nThese are descriptive differences for two seeds on one exposed development split,\nnot population estimates or independently confirmed effects. Matrix-decay versus\nno-decay is the controlled within-seed contrast; comparisons with P1-A0's seed 2003\nare not controlled optimizer comparisons.\n\n## Calibration learning curves, without best-checkpoint selection\n\n| Seed | Regimen | Update 0 | Update 1,200 | Update 3,000 | Update 6,000 |\n|---:|---|---:|---:|---:|---:|\n| 2017 | no_decay | 5.79% | 1.05% | 2.42% | 0.63% |\n| 2017 | matrix_decay | 5.79% | 28.74% | 94.84% | 99.58% |\n| 2027 | no_decay | 6.21% | 2.00% | 1.05% | 1.26% |\n| 2027 | matrix_decay | 6.21% | 4.84% | 89.37% | 98.00% |\n\nOnly update 6,000 determined readiness. Training and calibration accuracies for\nevery computation step and checkpoint are saved in the numeric outputs. A high\ntraining score cannot substitute for performance on unseen operand combinations.\nUniform random guessing has expected accuracy 1/17=5.88%; this is a mathematical\nreference, not a fitted baseline or a significance test.\n\n## Joint readiness decision\n\n| Regimen | Both seeds competent? | Common eligible steps | Ready for observer design? |\n|---|---|---|---|\n| no_decay | No | 2, 3, 4, 5 | No |\n| matrix_decay | Yes | 3, 4 | Yes |\n\nCompetence requires final calibration accuracy >=90% in **both** seeds. Target\nreadiness requires at least two common steps among 2..5, each with >=50 examples\nand >=25 distinct families in both decision-change classes, and each class at\nleast 10% of the examples. These are development screening rules, not an observer\npower analysis. The decision target is disagreement of current and final argmax,\nnot mathematical correctness, controller utility, or consciousness.\n\nThe selection rule prefers no-decay only if both regimens pass. No single model\nor favorable checkpoint can override the all-seed gate. Current status:\n`candidate_ready_for_observer_design`.\n\n## Data and interpretation limits\n\n- The models trained on 2,992 examples from 581 canonical operand families.\n- Calibration used 950 examples from 193 disjoint families; every permutation of\n  an operand triple remains in the same partition.\n- These calibration examples were already exposed in P1-A0 and influenced this\n  research direction. They remain unseen to gradient-based training, but are\n  **development data**, not fresh confirmation data.\n- The 195 reserved families were not expanded into examples or evaluated. Old\n  reserved model seeds 1103/1201/1301 and all closed T1/T2 weights were untouched.\n- A family can contribute to both decision-change classes through different\n  permutations. Permutations and computation steps are not independent models.\n- This tests only same-modulus, same-length operand combinations. No claim of\n  universal arithmetic ability or length extrapolation is established.\n\nNo history predictors were fitted. Therefore neither practical present-state\nsufficiency nor a training-induced reduction in history advantage was tested.\nExact continuation from a complete state is an architectural property, not a\nfinding of subjective awareness or inexpensive predictability.\n\n## Verification and provenance\n\nInherited method checks passed, together with exact zero-decay equivalence to the\noriginal Adam, synthetic closed-form decay/moment updates, the matrix/vector mask,\npairing of initializations and batches, and the all-seed selection logic. The\noriginal Transformer, autodiff and P1-A0 implementation were preserved unchanged.\n\nThe separate verification script recomputed all training and calibration logits\nfor all 16 saved checkpoints from the stored weights and inputs. Maximum absolute\nlogit difference was **0**. It also rechecked every accuracy,\nflip count, family count, gate, paired batch streams and initial weights, and the\nfinal decision. All 24,000 log entries were checked for consecutive update indices\nand finite loss/gradient norms; saved optimizer moments were checked for shape,\nfiniteness and step count. Training updates were not replayed. This is a numerical\naudit, not independent model replication.\n\nAt every checkpoint, resumption from each full current state was audited on 16\ncalibration examples at steps 0..6; the maximum recorded discrepancy was\n**0**. This resumption pays for the remaining Transformer steps and\ndoes not demonstrate that a cheaper predictor can match a history-aware one.\n\nScientific sources and protocol were locally frozen before natural training and\nremained unchanged. The source-hash record is an audit trail, not an independently\ntimestamped external registration. The verification script is post-run analysis\ncode and is identified separately from frozen scientific execution sources.\n\nLocal panel execution took **7.4 minutes**. Environment:\nPython 3.12.14, NumPy 2.3.5.\nNo GPU was used. Colab runtime can differ. An interactive Colab session was not\nexecuted here; notebook code cells were syntax-checked and embed the same locally\nexecuted source.\n\n## Next boundary\n\nThe prospectively frozen selection policy identifies `matrix_decay` as a development candidate. This supports designing the next observer protocol, but not automatically running probes or interpreting present-state sufficiency. Both seeds and every earlier checkpoint remain in the evidence record; a later confirmation study needs untouched evaluation conditions.\n\nBefore any observer fit, freeze the predictor families and budgets, strong\ncurrent-only baselines, matched/mismatched/permuted/residualized history controls,\ntrain-only preprocessing and eligibility, a useful-prediction criterion and\nnoninferiority margin, fitted-pipeline sensitivity, family/seed-aware uncertainty,\nand untouched confirmation rules. The prior 5% MSE threshold is not automatically\na valid Brier-score margin or deployment threshold. No automatic advancement\nwas triggered by this panel.\n\n## Files and reproduction\n\n- `P1_A0B_Colab.ipynb`: standalone NumPy notebook; method checks and this recorded\n  report by default. Full reproduction requires explicitly enabling its flag.\n- `P1_A0B_Protocol.md`: readable prospective specification.\n- `P1_A0B_Study.zip`: exact protocol/source, original-code provenance, all model\n  checkpoints, optimizer moments, paired batches, predictions, logs and audit.\n\nReproducing the same four exposed runs is not independent confirmation. Existing\nrun directories are never overwritten. Every scientific change requires a new\nversion; do not edit the completed protocol to relabel an observed outcome.\n"
(ROOT/'P1_A0B_Report.md').write_text(RECORDED_REPORT)
print('Prepared:',ROOT)
print('NumPy only; no model downloads or GPU required.')

## 1. Run method checks

Uses synthetic optimizer fixtures and untrained networks. Does not train the arithmetic panel or fit history predictors.

In [ ]:
method_output=ROOT/('methods_'+str(time.time_ns()))
subprocess.run([sys.executable,str(ROOT/'run_a0b.py'),'methods','--output',str(method_output)],check=True)
print('Methods passed:',json.loads((method_output/'method_checks.json').read_text())['passed'])

## 2. Recorded local results

These are the completed run's saved results, not a new Colab result.

In [ ]:
display(Markdown(RECORDED_REPORT))

## 3. Optional full reproduction

Enabling this flag runs four models × 6,000 updates using the frozen recipe, then verifies the results. Expect several CPU minutes; hardware affects duration. No extra runs or history probes will launch. This repeats exposed development work, not independent confirmation.

In [ ]:
RUN_REPRODUCTION=False
if RUN_REPRODUCTION:
    output=ROOT/('reproduction_'+str(time.time_ns()))
    subprocess.run([sys.executable,str(ROOT/'run_a0b.py'),'panel','--output',str(output)],check=True)
    subprocess.run([sys.executable,str(ROOT/'verify_a0b.py'),str(output)],check=True)
    rerun=json.loads((output/'result.json').read_text())
    print(json.dumps(rerun['decision'],indent=2))
else:
    print('No new training requested.')

## 4. Download this session's files

The separately delivered P1_A0B_Study.zip holds all original numerical evidence. This cell saves the sources, checks and any reproduction results created in this session.

In [ ]:
archive_path=ROOT.parent/(ROOT.name+'_outputs.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ROOT.rglob('*')):
        if path.is_file() and '__pycache__' not in path.parts:
            archive.write(path,path.relative_to(ROOT))
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('Outputs:',archive_path)